In [1]:
.libPaths(c("/workspace/.Rlib", .libPaths()))
library(flashier)
library(ebnm)

# Small simulated fit with real signal to inspect object structure
set.seed(1)
N <- 60; G <- 200; K <- 3
Z_true <- matrix(rnorm(N * K), N, K)
W_true <- matrix(rnorm(G * K, 0, 0.2), G, K)
W_true[sample(G * K, 30)] <- rnorm(30, 0, 3)  # sparse strong loadings
Y <- Z_true %*% t(W_true) + matrix(rnorm(N * G, 0, 0.5), N, G)
rownames(Y) <- paste0("obs", 1:N); colnames(Y) <- paste0("gene", 1:G)

f <- flash(Y, ebnm_fn = list(ebnm_normal, ebnm_point_normal),
           var_type = 2, greedy_Kmax = 10, backfit = TRUE, nullcheck = TRUE, verbose = 0)
cat("factors:", f$n_factors, "\n")
cat("names(f):", paste(names(f), collapse = ", "), "\n")
cat("dim L:", paste(dim(f$L), collapse = "x"), " dim F:", paste(dim(f$F), collapse = "x"), "\n")
cat("pve:", paste(round(f$pve, 3), collapse = ", "), "\n")
cat("residuals_sd length:", length(f$residuals_sd), "\n")
cat("names(f$flash_fit):", paste(names(f$flash_fit), collapse = ", "), "\n")
# fitted prior for loadings (mode 2)
str(f$flash_fit$ebnm_fn)
cat("---\n")
# check fitted g storage
if (!is.null(f$flash_fit$g)) str(f$flash_fit$g) else cat("no g in flash_fit\n")

Loading required package: ebnm



Loading required package: magrittr



Warning message:
“package ‘magrittr’ was built under R version 4.4.3”


factors: 3 


names(f): n_factors, pve, elbo, residuals_sd, L_pm, F_pm, L_psd, F_psd, L_lfsr, F_lfsr, L_ghat, F_ghat, sampler, flash_fit 


...

residuals_sd length: 200 


names(f$flash_fit): t.init, t.final, Y, Z, est.tau.dim, n.nonmissing, Y2, R2, est.tau, tau, obj, fix.dim, fix.idx, use.fixed.to.est.g, nonmissing.thresh, exclusions, verbose.lvl, verbose.fns, verbose.colnames, verbose.colwidths, conv.crit.fn, conv.tol, warmstart.greedy, EF, EF2, KL, g, ebnm.fn, is.zero, is.valid, warmstart.backfits 


 NULL


---


List of 3
 $ :List of 2
  ..$ :List of 3
  .. ..$ pi  : num 1
  .. ..$ mean: num 0
  .. ..$ sd  : num 1.26
  .. ..- attr(*, "class")= chr "normalmix"
  .. ..- attr(*, "row.names")= int 1
  ..$ :List of 3
  .. ..$ pi  : num [1:2] 0.335 0.665
  .. ..$ mean: num [1:2] 0 0
  .. ..$ sd  : num [1:2] 0 0.728
  .. ..- attr(*, "class")= chr "normalmix"
  .. ..- attr(*, "row.names")= int [1:2] 1 2
 $ :List of 2
  ..$ :List of 3
  .. ..$ pi  : num 1
  .. ..$ mean: num 0
  .. ..$ sd  : num 1.14
  .. ..- attr(*, "class")= chr "normalmix"
  .. ..- attr(*, "row.names")= int 1
  ..$ :List of 3
  .. ..$ pi  : num [1:2] 0.44 0.56
  .. ..$ mean: num [1:2] 0 0
  .. ..$ sd  : num [1:2] 0 0.836
  .. ..- attr(*, "class")= chr "normalmix"
  .. ..- attr(*, "row.names")= int [1:2] 1 2
 $ :List of 2
  ..$ :List of 3
  .. ..$ pi  : num 1
  .. ..$ mean: num 0
  .. ..$ sd  : num 0.971
  .. ..- attr(*, "class")= chr "normalmix"
  .. ..- attr(*, "row.names")= int 1
  ..$ :List of 3
  .. ..$ pi  : num [1:2] 0.476 0.52

In [2]:
# Check flash() signature and init conventions
print(args(flashier::flash))
cat("---\n")
cat(exists("flash_init", where = asNamespace("flashier")), "flash_init exists\n")
print(args(flashier::flash_init))
cat("---\n")
# dims of posterior means
f2 <- flash(Y, ebnm_fn = list(ebnm_normal, ebnm_point_normal), var_type = 2,
            greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0)
cat("L_pm:", paste(dim(f2$L_pm), collapse="x"), " F_pm:", paste(dim(f2$F_pm), collapse="x"), "\n")

# Test custom ebnm_fn closure with fixed prior (needed for L1 shared priors)
pooled_g <- ebnm::ebnm_point_normal(as.vector(f2$F_pm))$fitted_g
cat("pooled prior: pi0 =", round(1 - pooled_g$pi[2], 3), " slab sd =", round(pooled_g$sd[2], 3), "\n")
my_fn <- function(x, s, g, fixg, output, ...) {
  ebnm::ebnm_point_normal(x = x, s = s, g = pooled_g, fixg = TRUE, output = output)
}
f3 <- tryCatch(
  flash(Y, ebnm_fn = list(ebnm_normal, my_fn), var_type = 2,
        greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f3)) cat(f3, "\n") else cat("custom ebnm_fn OK, factors:", f3$n_factors, "\n")

# Test init from existing loadings (needed for L2)
init_obj <- flashier::flash_init(f2)
f4 <- tryCatch(
  flash(Y, init = init_obj, ebnm_fn = list(ebnm_normal, ebnm_point_normal),
        var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f4)) cat(f4, "\n") else cat("init from flash object OK, factors:", f4$n_factors, "\n")

# Missing data support check (needed for block-E)
Yna <- Y; Yna[1:10, 1:50] <- NA
f5 <- tryCatch(
  flash(Yna, ebnm_fn = list(ebnm_normal, ebnm_point_normal), var_type = 2,
        greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f5)) cat(f5, "\n") else cat("missing data OK, factors:", f5$n_factors, "\n")

function (data, S = NULL, ebnm_fn = ebnm_point_normal, var_type = 0L, 
    greedy_Kmax = 50L, backfit = FALSE, nullcheck = TRUE, verbose = 1L) 
NULL


---


TRUE flash_init exists


function (data, S = NULL, var_type = 0L, S_dim = NULL) 
NULL


---


L_pm: 60x3  F_pm: 200x3 


pooled prior: pi0 = 1  slab sd = 1 


ERROR: Argument to ebnm.fn is incorrectly specified. See ?flash_ebnm for details. 


ERROR: Error in must.be.supported.data.type(data, allow.null = FALSE, allow.lowrank = TRUE): Invalid argument to data.


In [3]:
# flash_ebnm helper for custom priors
print(args(flashier::flash_ebnm))
cat("---\n")
# Try fixed-prior via flash_ebnm
f3 <- tryCatch(
  flash(Y, ebnm_fn = list(flash_ebnm(ebnm_normal),
                          flash_ebnm(ebnm_point_normal, g = pooled_g, fixg = TRUE)),
        var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f3)) cat(f3, "\n") else cat("fixed-prior flash_ebnm OK, factors:", f3$n_factors, "\n")
cat("---\n")
# Pipeline API for initialization (L2)
cat(grep("^flash_", ls(asNamespace("flashier")), value = TRUE), sep = "\n")

function (...) 
NULL


---


ERROR: Argument to ebnm.fn is incorrectly specified. See ?flash_ebnm for details. 


---


flash_add_intercept
flash_backfit
flash_clear_timeout
flash_conv_crit_elbo_diff
flash_conv_crit_max_chg
flash_conv_crit_max_chg_F
flash_conv_crit_max_chg_L
flash_ebnm
flash_factors_fix
flash_factors_init
flash_factors_remove
flash_factors_reorder
flash_factors_set_to_zero
flash_factors_unfix
flash_fit
flash_fit_get_elbo
flash_fit_get_est_tau
flash_fit_get_fixed_tau
flash_fit_get_g
flash_fit_get_KL
flash_fit_get_p2m
flash_fit_get_pm
flash_fit_get_tau
flash_greedy
flash_greedy_init_default
flash_greedy_init_irlba
flash_greedy_init_softImpute
flash_init
flash_nullcheck
flash_set_conv_crit
flash_set_timeout
flash_set_verbose
flash_verbose_elbo
flash_verbose_elbo_diff
flash_verbose_max_chg
flash_verbose_max_chg_F
flash_verbose_max_chg_L


In [4]:
# Read flash_ebnm source and its validator
print(flashier::flash_ebnm)
cat("===\n")
ns <- asNamespace("flashier")
# find the ebnm spec handler
src <- capture.output(print(get("handle_ebnm_fn", envir = ns)))
cat(paste(head(src, 60), collapse = "\n"))

function (...) 
{
    args <- list(...)
    if (any(c("x", "s", "output") %in% names(args))) {
        stop("'x', 's', and 'output' must not be supplied as arguments to ", 
            "'flash_ebnm'.")
    }
    if (any(c("g_init", "fix_g") %in% names(args))) {
        if (is.null(args$g_init)) {
            stop("If 'fix_g' is supplied as an argument to flash_ebnm, then ", 
                "'g_init' must be as well.")
        }
        if (is.null(args$fix_g) || !args$fix_g) {
            stop("If 'g_init' is supplied as an argument to flash_ebnm, then it ", 
                "must be fixed (i.e., 'fix_g = TRUE' should be included as an ", 
                "argument). If you were trying to fix the grid of the prior, then ", 
                "use parameter 'scale' rather than 'g_init'.")
        }
        if (!is.null(args$group)) {
            ebnm.fn <- function(x, s, g_init, fix_g, output) {
                if (identical(x, c(-1, 0, 1))) {
                  x <- rep(x, length.out = l

===


ERROR: Error in get("handle_ebnm_fn", envir = ns): object 'handle_ebnm_fn' not found


In [5]:
# 1) Fixed prior via flash_ebnm(g_init, fix_g = TRUE)
f3 <- tryCatch(
  flash(Y, ebnm_fn = list(flash_ebnm(prior_family = "normal"),
                          flash_ebnm(prior_family = "point_normal", g_init = pooled_g, fix_g = TRUE)),
        var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f3)) cat(f3, "\n") else cat("fixed-prior OK, factors:", f3$n_factors, "\n")

# 2) Pipeline init flow for L2
print(args(flashier::flash_factors_init))
cat("---\n")
# donor loadings: use f2's F_pm as 'donor programs', initialize a new fit with them
L_init <- matrix(0, nrow(Y), f2$n_factors)  # scores will be estimated
F_init <- f2$F_pm
f4 <- tryCatch(
  flash_init(Y, var_type = 2) |>
    flash_factors_init(list(L_init, F_init),
                       ebnm_fn = list(flash_ebnm(prior_family = "normal"),
                                      flash_ebnm(prior_family = "point_normal"))) |>
    flash_backfit(ebnm_fn = list(flash_ebnm(prior_family = "normal"),
                                 flash_ebnm(prior_family = "point_normal")), maxiter = 100, verbose = 0) |>
    flash_nullcheck(ebnm_fn = list(flash_ebnm(prior_family = "normal"),
                                   flash_ebnm(prior_family = "point_normal")), verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f4)) cat(f4, "\n") else {
  cat("pipeline init OK, class:", class(f4), " factors:", f4$n_factors, "\n")
  cat("has L_pm:", !is.null(f4$L_pm), "\n")
}

ERROR: missing value where TRUE/FALSE needed 


function (flash, init, ebnm_fn = ebnm_point_normal) 
NULL


---


ERROR: unused argument (ebnm_fn = list(flash_ebnm(prior_family = "normal"), flash_ebnm(prior_family = "point_normal"))) 


In [6]:
print(args(flashier::flash_backfit))
print(args(flashier::flash_greedy))
print(args(flashier::flash_nullcheck))
cat("===\n")
# Debug fixed-prior error with traceback
options(error = function() traceback(2))
f3 <- tryCatch(
  flash(Y, ebnm_fn = list(flash_ebnm(prior_family = "normal"),
                          flash_ebnm(prior_family = "point_normal", g_init = pooled_g, fix_g = TRUE)),
        var_type = 2, greedy_Kmax = 5, backfit = FALSE, nullcheck = FALSE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
options(error = NULL)
if (is.character(f3)) cat(f3, "\n") else cat("fixed-prior (no backfit/nullcheck) OK, factors:", f3$n_factors, "\n")
# check pooled_g structure
str(pooled_g)

function (flash, kset = NULL, extrapolate = TRUE, warmstart = TRUE, 
    maxiter = 500, tol = NULL, verbose = NULL) 
NULL


function (flash, Kmax = 1, ebnm_fn = ebnm_point_normal, init_fn = NULL, 
    extrapolate = FALSE, warmstart = FALSE, maxiter = 500, tol = NULL, 
    verbose = NULL) 
NULL


function (flash, kset = NULL, remove = TRUE, tol = NULL, verbose = NULL) 
NULL


===


ERROR: missing value where TRUE/FALSE needed 


List of 3
 $ pi  : num [1:2] 1 0
 $ mean: num [1:2] 0 0
 $ sd  : num [1:2] 0 1
 - attr(*, "class")= chr "normalmix"
 - attr(*, "row.names")= int [1:2] 1 2


In [7]:
# L1: pool fitted priors by mixture averaging across factors
pool_priors <- function(g_list) {
  # g_list: list of normalmix objects (per-factor loading priors)
  pis <- unlist(lapply(g_list, function(g) g$pi))
  sds <- unlist(lapply(g_list, function(g) g$sd))
  mns <- unlist(lapply(g_list, function(g) g$mean))
  w <- pis / length(g_list)  # average the mixtures
  keep <- w > 1e-12
  ashr::normalmix(pi = w[keep], mean = mns[keep], sd = sds[keep])
}
gs <- lapply(f2$flash_fit$g, function(x) x[[2]])  # gene-side prior per factor
shared_g <- pool_priors(gs)
cat("pooled prior components:", length(shared_g$pi), " slab weight:",
    round(sum(shared_g$pi[shared_g$sd > 1e-8]), 3), "\n")

f3 <- tryCatch(
  flash(Y, ebnm_fn = list(flash_ebnm(prior_family = "normal"),
                          flash_ebnm(prior_family = "point_normal", g_init = shared_g, fix_g = TRUE)),
        var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f3)) cat(f3, "\n") else cat("L1 fixed-prior fit OK, factors:", f3$n_factors, "\n")

# L2: pipeline flow with donor initialization + greedy addition
ebnm_list <- list(flash_ebnm(prior_family = "normal"), flash_ebnm(prior_family = "point_normal"))
f4 <- tryCatch(
  flash_init(Y, var_type = 2) |>
    flash_factors_init(list(L_init, F_init), ebnm_fn = ebnm_list) |>
    flash_greedy(Kmax = 3, ebnm_fn = ebnm_list, verbose = 0) |>
    flash_backfit(maxiter = 100, verbose = 0) |>
    flash_nullcheck(verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f4)) cat(f4, "\n") else cat("L2 pipeline OK, class:", class(f4)[1], " factors:", f4$n_factors, "\n")

pooled prior components: 6  slab weight: 0.583 


ERROR: Argument to ebnm.fn is incorrectly specified. See ?flash_ebnm for details. 


  --Estimate of factor 1 is numerically zero!
  --Estimate of factor 2 is numerically zero!
  --Estimate of factor 3 is numerically zero!


L2 pipeline OK, class: flash  factors: 3 


In [8]:
ns <- asNamespace("flashier")
cands <- grep("ebnm", ls(ns), value = TRUE)
print(cands)
# look for the validator
for (nm in cands) {
  fn <- get(nm, envir = ns)
  if (is.function(fn)) {
    src <- paste(deparse(body(fn)), collapse = "\n")
    if (grepl("incorrectly specified", src)) {
      cat("=== validator:", nm, "===\n")
      cat(src, "\n")
    }
  }
}

 [1] "add.factor.to.ebnm.fn"  "add.fixed.to.ebnm.args" "calc.all.ebnm.args"    
 [4] "calc.ebnm.args"         "default.ebnm.output"    "ebnm.nowarn"           
 [7] "extend.ebnm.lists"      "flash_ebnm"             "get.ebnm.fn"           
[10] "get.ebnm.fn.k"          "get.tau.for.ebnm.calc"  "handle.ebnm.fn"        
[13] "parallel.ebnm.fn"       "set.ebnm.fn"            "solve.ebnm"            
[16] "solve.ebnm.parallel"    "test.ebnm.fn"          


=== validator: handle.ebnm.fn ===
{
    error.msg <- paste("Argument to ebnm.fn is incorrectly specified. See ?flash_ebnm for details.")
    if (length(ebnm.fn) == 1) 
        ebnm.fn <- rep(list(ebnm.fn), data.dim)
    if (length(ebnm.fn) != data.dim) 
        stop(error.msg)
    dim.signs <- sapply(ebnm.fn, test.ebnm.fn, error.msg)
    return(list(ebnm.fn = ebnm.fn, dim.signs = dim.signs))
} 


In [9]:
ns <- asNamespace("flashier")
cat(paste(deparse(body(get("test.ebnm.fn", envir = ns))), collapse = "\n"))

{
    tryCatch(tmp <- ebnm.fn(x = c(-10, 0, 10), s = rep(1, 3), 
        g_init = NULL, fix_g = FALSE, output = default.ebnm.output), 
        error = function(e) stop(error.msg))
    if (all(coef(tmp) >= 0)) {
        return(1)
    }
    else if (all(coef(tmp) <= 0)) {
        return(-1)
    }
    else {
        return(0)
    }
}

In [10]:
# Hand-written closure: ignore validator's g_init/fix_g, always use shared_g
make_fixed_prior_fn <- function(g, family = "point_normal") {
  force(g); force(family)
  function(x, s, g_init, fix_g, output) {
    ebnm::ebnm(x, s, prior_family = family, g_init = g, fix_g = TRUE, output = output)
  }
}
fn_fixed <- make_fixed_prior_fn(shared_g)
# validator trial call
tmp <- fn_fixed(x = c(-10, 0, 10), s = rep(1, 3), g_init = NULL, fix_g = FALSE,
                output = get("default.ebnm.output", envir = asNamespace("flashier")))
cat("trial call OK, coef:", paste(round(coef(tmp), 3), collapse = ", "), "\n")

f3 <- tryCatch(
  flash(Y, ebnm_fn = list(flash_ebnm(prior_family = "normal"), fn_fixed),
        var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f3)) cat(f3, "\n") else {
  cat("L1 shared-prior fit OK, factors:", f3$n_factors, "\n")
  # compare recovery with free-prior fit
  cat("cor(W_true direction, f2):", round(max(abs(cor(f2$F_pm, W_true))), 3),
      " f3:", round(max(abs(cor(f3$F_pm, W_true))), 3), "\n")
}

ERROR: Error in check_g_init(g_init = g_init, fix_g = fix_g, mode = mode, scale = scale, : g_init does not have the correct number of components.


In [11]:
make_fixed_prior_fn <- function(g) {
  force(g)
  function(x, s, g_init, fix_g, output) {
    ebnm::ebnm(x, s, prior_family = "normal_mixture", g_init = g, fix_g = TRUE, output = output)
  }
}
fn_fixed <- make_fixed_prior_fn(shared_g)
f3 <- tryCatch(
  flash(Y, ebnm_fn = list(flash_ebnm(prior_family = "normal"), fn_fixed),
        var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0),
  error = function(e) paste("ERROR:", conditionMessage(e))
)
if (is.character(f3)) cat(f3, "\n") else {
  cat("L1 shared-prior fit OK, factors:", f3$n_factors, "\n")
  rec2 <- max(abs(cor(f2$F_pm, W_true))); rec3 <- max(abs(cor(f3$F_pm, W_true)))
  cat("best loading recovery cor  free-prior:", round(rec2, 3), " shared-prior:", round(rec3, 3), "\n")
}

ERROR: Argument to ebnm.fn is incorrectly specified. See ?flash_ebnm for details. 


In [12]:
deo <- get("default.ebnm.output", envir = asNamespace("flashier"))
print(deo)
trial <- tryCatch(
  fn_fixed(x = c(-10, 0, 10), s = rep(1, 3), g_init = NULL, fix_g = FALSE, output = deo),
  error = function(e) paste("TRIAL ERROR:", conditionMessage(e))
)
if (is.character(trial)) cat(trial, "\n") else { cat("trial OK\n"); print(coef(trial)) }

[1] "posterior_mean"          "posterior_second_moment"
[3] "fitted_g"                "log_likelihood"         


TRIAL ERROR: 'arg' should be one of “point_normal”, “point_laplace”, “point_exponential”, “normal”, “horseshoe”, “normal_scale_mixture”, “unimodal”, “unimodal_symmetric”, “unimodal_nonnegative”, “unimodal_nonpositive”, “generalized_binary”, “npmle”, “deconvolver”, “flat”, “point_mass”, “ash” 


In [13]:
for (fam in c("normal_scale_mixture", "npmle")) {
  res <- tryCatch(
    ebnm::ebnm(c(-10, 0, 10), rep(1, 3), prior_family = fam, g_init = shared_g, fix_g = TRUE,
               output = c("posterior_mean", "posterior_second_moment", "fitted_g", "log_likelihood")),
    error = function(e) paste("ERROR:", conditionMessage(e))
  )
  if (is.character(res)) cat(fam, ":", res, "\n") else cat(fam, ": OK, posterior means:", paste(round(res$posterior$mean, 3), collapse = ", "), "\n")
}

normal_scale_mixture : OK, posterior means: -4.062, 0, 4.062 
npmle : OK, posterior means: -4.062, 0, 4.062 


In [14]:
make_fixed_prior_fn <- function(g) {
  force(g)
  function(x, s, g_init, fix_g, output) {
    ebnm::ebnm(x, s, prior_family = "normal_scale_mixture", g_init = g, fix_g = TRUE, output = output)
  }
}
fn_fixed <- make_fixed_prior_fn(shared_g)
f3 <- flash(Y, ebnm_fn = list(flash_ebnm(prior_family = "normal"), fn_fixed),
            var_type = 2, greedy_Kmax = 5, backfit = TRUE, nullcheck = TRUE, verbose = 0)
cat("L1 shared-prior fit OK, factors:", f3$n_factors, "\n")
cat("recovery cor  free:", round(max(abs(cor(f2$F_pm, W_true))), 3),
    " shared:", round(max(abs(cor(f3$F_pm, W_true))), 3), "\n")

# Also verify L2 with proper L initialization (regression of Y on donor loadings)
L_init2 <- Y %*% F_init %*% MASS::ginv(crossprod(F_init) + 1e-6 * diag(ncol(F_init)))
f4 <- flash_init(Y, var_type = 2) |>
  flash_factors_init(list(L_init2, F_init), ebnm_fn = list(flash_ebnm(prior_family = "normal"), flash_ebnm(prior_family = "point_normal"))) |>
  flash_greedy(Kmax = 2, ebnm_fn = list(flash_ebnm(prior_family = "normal"), flash_ebnm(prior_family = "point_normal")), verbose = 0) |>
  flash_backfit(maxiter = 100, verbose = 0) |>
  flash_nullcheck(verbose = 0)
cat("L2 pipeline factors:", f4$n_factors, " recovery:", round(max(abs(cor(f4$F_pm, W_true))), 3), "\n")
cat("\nAll flashier interface patterns verified.\n")

L1 shared-prior fit OK, factors: 3 


recovery cor  free: 0.995  shared: 0.995 


L2 pipeline factors: 3  recovery: 0.995 



All flashier interface patterns verified.


In [15]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

sim <- simulate_programs(scenario = 4, N = 120, G = 400, seed = 42)
fit <- fit_celltype_programs(sim$x, max_factors = 10, seed = 42) |>
  canonicalize_programs()
print(fit)
cat("\n")
print(program_summary(fit))

cell_program_fit
    B             2 programs  (pve: 0.22, 0.13)
    CD4           1 programs  (pve: 0.23)
    CD8           2 programs  (pve: 0.27, 0.23)
    NK            2 programs  (pve: 0.33, 0.18)
    Mono          2 programs  (pve: 0.27, 0.32)


  cell_type n_obs n_genes n_programs        R2      elbo
1         B   120     400          2 0.3553178 -68649.13
2       CD4   120     400          1 0.2344894 -68239.71
3       CD8   120     400          2 0.5011915 -68601.19
4        NK   120     400          2 0.5013868 -68446.83
5      Mono   120     400          2 0.5925444 -68405.09


In [16]:
## Recovery: correlate each fitted program's scores with true activities
true_acts <- sim$truth$activities
Z <- program_scores(fit, "wide")
rec <- do.call(rbind, lapply(names(true_acts), function(pn) {
  z <- true_acts[[pn]]$z
  cors <- abs(stats::cor(Z[rownames(sim$x$matrices[[1]]), ], z))
  best <- which.max(cors)
  data.frame(true_program = pn, type = true_acts[[pn]]$type,
             best_fit = colnames(Z)[best], cor = round(cors[best], 3))
}))
print(rec)

cat("\n=== L4 sharing spectrum ===\n")
ss <- sharing_spectrum(fit, n_perm = 100, seed = 1)
print(ss$summary)

  true_program    type   best_fit   cor
1       B_priv private    B__F001 0.996
2    Mono_priv private Mono__F001 0.998
3 CD8NK_shared partial  CD8__F001 0.997
4   IFN_global  global   NK__F001 0.998



=== L4 sharing spectrum ===


                   pair k_a k_b G_intersect min_angle_deg n_shared_05
B vs CD4       B vs CD4   2   1         400          85.2           0
B vs CD8       B vs CD8   2   2         400          85.0           0
B vs NK         B vs NK   2   2         400          87.3           0
B vs Mono     B vs Mono   2   2         400          84.2           0
CD4 vs CD8   CD4 vs CD8   1   2         400          87.4           0
CD4 vs NK     CD4 vs NK   1   2         400          85.7           0
CD4 vs Mono CD4 vs Mono   1   2         400          90.0           0
CD8 vs NK     CD8 vs NK   2   2         400          80.5           0
CD8 vs Mono CD8 vs Mono   2   2         400          78.6           0
NK vs Mono   NK vs Mono   2   2         400          87.0           0


In [17]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

sim <- simulate_programs(scenario = 4, N = 120, G = 400, seed = 42)
fit <- fit_celltype_programs(sim$x, max_factors = 10, seed = 42) |>
  canonicalize_programs()

cat("=== Scores-space sharing spectrum (activity coordination) ===\n")
ss <- sharing_spectrum(fit, space = "scores", n_perm = 100, seed = 1)
print(ss$summary)

cat("\n=== Loadings-space spectrum (shared gene usage) ===\n")
ss2 <- sharing_spectrum(fit, space = "loadings", n_perm = 100, seed = 1)
print(ss2$summary)

=== Scores-space sharing spectrum (activity coordination) ===


                   pair k_a k_b effective_dim min_angle_deg n_shared_05
B vs CD4       B vs CD4   2   1           120           7.9           1
B vs CD8       B vs CD8   2   2           120           7.6           1
B vs NK         B vs NK   2   2           120           7.6           1
B vs Mono     B vs Mono   2   2           120           7.6           1
CD4 vs CD8   CD4 vs CD8   1   2           120           7.0           1
CD4 vs NK     CD4 vs NK   1   2           120           6.0           1
CD4 vs Mono CD4 vs Mono   1   2           120           6.4           1
CD8 vs NK     CD8 vs NK   2   2           120           5.2           2
CD8 vs Mono CD8 vs Mono   2   2           120           5.6           1
NK vs Mono   NK vs Mono   2   2           120           5.1           1



=== Loadings-space spectrum (shared gene usage) ===


                   pair k_a k_b effective_dim min_angle_deg n_shared_05
B vs CD4       B vs CD4   2   1           400          85.2           0
B vs CD8       B vs CD8   2   2           400          85.0           0
B vs NK         B vs NK   2   2           400          87.3           0
B vs Mono     B vs Mono   2   2           400          84.2           0
CD4 vs CD8   CD4 vs CD8   1   2           400          87.4           0
CD4 vs NK     CD4 vs NK   1   2           400          85.7           0
CD4 vs Mono CD4 vs Mono   1   2           400          90.0           0
CD8 vs NK     CD8 vs NK   2   2           400          80.5           0
CD8 vs Mono CD8 vs Mono   2   2           400          78.6           0
NK vs Mono   NK vs Mono   2   2           400          87.0           0


In [18]:
run_scenario <- function(sc, ...) {
  sim <- simulate_programs(scenario = sc, ...)
  fit <- fit_celltype_programs(sim$x, max_factors = 10, seed = 42) |> canonicalize_programs()
  ss <- sharing_spectrum(fit, space = "scores", n_perm = 100, seed = 1)
  list(sim = sim, fit = fit, ss = ss)
}

cat("=== Scenario 1 (all private): expect ~0 shared ===\n")
r1 <- run_scenario(1, N = 120, G = 400, seed = 42)
print(r1$ss$summary[, c("pair", "min_angle_deg", "n_shared_05")])

cat("\n=== Scenario 7 (same genes, independent activities): scores-space must stay null ===\n")
r7 <- run_scenario(7, N = 120, G = 400, seed = 42)
print(r7$ss$summary[, c("pair", "min_angle_deg", "n_shared_05")])
cat("-- loadings space (should detect the shared IFN-like genes in B vs Mono):\n")
ss7l <- sharing_spectrum(r7$fit, space = "loadings", n_perm = 100, seed = 1)
print(ss7l$summary[, c("pair", "min_angle_deg", "n_shared_05")])

cat("\n=== Scenario 5 (correlated but distinct, cor=0.7): documented confound ===\n")
r5 <- run_scenario(5, N = 120, G = 400, seed = 42)
print(r5$ss$summary[, c("pair", "min_angle_deg", "n_shared_05")])

=== Scenario 1 (all private): expect ~0 shared ===


                   pair min_angle_deg n_shared_05
B vs CD4       B vs CD4          83.3           0
B vs CD8       B vs CD8          78.0           0
B vs NK         B vs NK          84.5           0
B vs Mono     B vs Mono          76.9           0
CD4 vs CD8   CD4 vs CD8          78.8           0
CD4 vs NK     CD4 vs NK          82.4           0
CD4 vs Mono CD4 vs Mono          83.0           0
CD8 vs NK     CD8 vs NK          77.2           0
CD8 vs Mono CD8 vs Mono          87.1           0
NK vs Mono   NK vs Mono          81.1           0



=== Scenario 7 (same genes, independent activities): scores-space must stay null ===


                   pair min_angle_deg n_shared_05
B vs CD4       B vs CD4          87.0           0
B vs CD8       B vs CD8          86.0           0
B vs NK         B vs NK          87.7           0
B vs Mono     B vs Mono          87.9           0
CD4 vs CD8   CD4 vs CD8          89.1           0
CD4 vs NK     CD4 vs NK          82.5           0
CD4 vs Mono CD4 vs Mono          88.2           0
CD8 vs NK     CD8 vs NK          81.9           0
CD8 vs Mono CD8 vs Mono          85.9           0
NK vs Mono   NK vs Mono          83.4           0


-- loadings space (should detect the shared IFN-like genes in B vs Mono):


                   pair min_angle_deg n_shared_05
B vs CD4       B vs CD4          90.0           0
B vs CD8       B vs CD8          87.8           0
B vs NK         B vs NK          86.6           0
B vs Mono     B vs Mono          82.3           1
CD4 vs CD8   CD4 vs CD8          88.7           0
CD4 vs NK     CD4 vs NK          90.0           0
CD4 vs Mono CD4 vs Mono          90.0           0
CD8 vs NK     CD8 vs NK          90.0           0
CD8 vs Mono CD8 vs Mono          89.0           0
NK vs Mono   NK vs Mono          87.9           0



=== Scenario 5 (correlated but distinct, cor=0.7): documented confound ===


                   pair min_angle_deg n_shared_05
B vs CD4       B vs CD4          82.5           0
B vs CD8       B vs CD8          88.2           0
B vs NK         B vs NK          80.9           1
B vs Mono     B vs Mono          45.1           1
CD4 vs CD8   CD4 vs CD8          81.9           0
CD4 vs NK     CD4 vs NK          83.9           0
CD4 vs Mono CD4 vs Mono          83.5           0
CD8 vs NK     CD8 vs NK          80.2           0
CD8 vs Mono CD8 vs Mono          87.8           0
NK vs Mono   NK vs Mono          81.7           0


In [19]:
fit4 <- fit  ## Scenario 4 fit from earlier

## ICA integration
fit4 <- integrate_programs(fit4, method = "ica", n_states = 4, n_runs = 10, seed = 7)
cat("=== ICA ===\n"); print(integration_summary(fit4))
## does an IC recover the global IFN activity?
z_ifn <- sim$truth$activities$IFN_global$z
ic_cors <- abs(stats::cor(fit4$integration$states, z_ifn[match(rownames(fit4$integration$states), paste0("obs", 1:120))]))
cat("max |cor| of ICs with true IFN_global activity:", round(max(ic_cors), 3), "\n")

## PVA integration
fit4 <- integrate_programs(fit4, method = "pva", n_perm = 100, seed = 7)
cat("\n=== PVA ===\n"); print(integration_summary(fit4))
if (!is.null(fit4$integration$states)) {
  pv_cors <- abs(stats::cor(fit4$integration$states, z_ifn[paste0("obs", 1:120)], use = "pairwise"))
  cat("max |cor| of PV axes with true IFN_global:", round(max(pv_cors, na.rm = TRUE), 3), "\n")
  z_cd8nk <- sim$truth$activities$CD8NK_shared$z
  pv_cors2 <- abs(stats::cor(fit4$integration$states, z_cd8nk[paste0("obs", 1:120)], use = "pairwise"))
  cat("max |cor| of PV axes with true CD8NK_shared:", round(max(pv_cors2, na.rm = TRUE), 3), "\n")
}

## trace_state: gene traceability of the top PVA axis
st <- fit4$integration$summary$state[1]
tr <- trace_state(fit4, state = st, top_n = 5)
cat("\ntrace_state(", st, ") top genes per cell type:\n")
print(tr)

=== ICA ===


     method state stability
IC01    ica  IC01 0.9712588
IC02    ica  IC02 0.8168462
IC03    ica  IC03 0.9812086
IC04    ica  IC04 0.8244537


max |cor| of ICs with true IFN_global activity: 0.806 



=== PVA ===


  method state   pair    cosine    p_value
1    pva  PV01 B-Mono 0.9913235 0.00990099
2    pva  PV02   B-NK 0.9912104 0.00990099
3    pva  PV03  B-CD8 0.9911277 0.00990099
4    pva  PV04  B-CD4 0.9904144 0.00990099


Warning message in max(pv_cors, na.rm = TRUE):
“no non-missing arguments to max; returning -Inf”


max |cor| of PV axes with true IFN_global: -Inf 


Warning message in max(pv_cors2, na.rm = TRUE):
“no non-missing arguments to max; returning -Inf”


max |cor| of PV axes with true CD8NK_shared: -Inf 


ERROR: Error: 


In [20]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)
fit4 <- integrate_programs(fit4, method = "pva", n_perm = 100, seed = 7)
summ <- integration_summary(fit4)
print(summ)
z_ifn <- sim$truth$activities$IFN_global$z
z_cd8nk <- sim$truth$activities$CD8NK_shared$z
pv_ifn <- abs(stats::cor(fit4$integration$states, z_ifn[paste0("obs", 1:120)], use = "pairwise"))
pv_cd8nk <- abs(stats::cor(fit4$integration$states, z_cd8nk[paste0("obs", 1:120)], use = "pairwise"))
cat("max |cor| PV axes vs IFN_global:", round(max(pv_ifn), 3), "\n")
cat("max |cor| PV axes vs CD8NK_shared:", round(max(pv_cd8nk), 3), "\n")

## trace_state on the top axis
st <- summ$state[1]
tr <- trace_state(fit4, state = st, top_n = 5)
cat("\ntrace_state(", st, "):\n")
print(tr)
## check against true IFN loadings in B
w_true <- sim$truth$loadings$B$IFN_global
top_true <- names(sort(-abs(w_true)))[1:10]
cat("\ntrue top-10 IFN genes in B:", paste(top_true, collapse = ", "), "\n")

  method state   pair    cosine    p_value
1    pva  PV01 B-Mono 0.9913235 0.00990099
2    pva  PV02   B-NK 0.9912104 0.00990099
3    pva  PV03  B-CD8 0.9911277 0.00990099
4    pva  PV04  B-CD4 0.9904144 0.00990099


max |cor| PV axes vs IFN_global: NA 


max |cor| PV axes vs CD8NK_shared: NA 



trace_state( PV01 ):


$B
      g312       g193        g19       g102       g343 
-0.4241344  0.3215346  0.2307472  0.1979772 -0.1832206 

$Mono
       g43        g41       g227        g94        g10 
-0.8715597 -0.8643838 -0.5276292 -0.3688947  0.3652238 




true top-10 IFN genes in B: g312, g193, g19, g102, g343, g167, g120, g373, g165, g151 


In [21]:
integ <- fit4$integration
str(integ$states[1:3, 1:2])
ax <- integ$axes[[1]]
cat("pair:", ax$pair, " direction:", ax$direction, "\n")
cat("w_ref names:", paste(head(names(ax$w_ref), 3), collapse=", "), " len:", length(ax$w_ref), "\n")
Z <- program_scores(fit4, "wide")
cat("Z dim:", paste(dim(Z), collapse="x"), "\n")
cat("w_ref names in colnames(Z):", paste(names(ax$w_ref) %in% colnames(Z), collapse=", "), "\n")
progs <- names(ax$w_ref)[names(ax$w_ref) %in% colnames(Z)]
proj <- as.numeric(Z[, progs, drop = FALSE] %*% ax$w_ref[progs])
cat("proj summary:", paste(round(summary(proj), 3), collapse=", "), "\n")
cat("n_contrib col 1:", paste(unique(integ$n_contributing[,1]), collapse=","), "\n")

 num [1:3, 1:2] -0.1151 0.0955 -0.1051 -0.1071 0.0962 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:3] "obs1" "obs2" "obs3"
  ..$ : chr [1:2] "PV01" "PV02"


pair: B Mono  direction: 1 


w_ref names: B__F001, B__F002  len: 2 


Z dim: 120x9 


w_ref names in colnames(Z): TRUE, TRUE 


proj summary: -0.237, -0.071, -0.01, 0, 0.069, 0.211 


n_contrib col 1: 2 


In [22]:
z_ifn <- as.numeric(sim$truth$activities$IFN_global$z)
z_cd8nk <- as.numeric(sim$truth$activities$CD8NK_shared$z)
obs_order <- match(rownames(fit4$integration$states), paste0("obs", 1:120))
pv_ifn <- abs(stats::cor(fit4$integration$states, z_ifn[obs_order], use = "pairwise"))
pv_cd8nk <- abs(stats::cor(fit4$integration$states, z_cd8nk[obs_order], use = "pairwise"))
summ <- integration_summary(fit4)
res <- data.frame(state = summ$state, pair = summ$pair,
                  cor_IFN_global = round(as.numeric(pv_ifn), 3),
                  cor_CD8NK = round(as.numeric(pv_cd8nk), 3))
print(res)

  state   pair cor_IFN_global cor_CD8NK
1  PV01 B-Mono          0.998     0.130
2  PV02   B-NK          0.998     0.133
3  PV03  B-CD8          0.997     0.133
4  PV04  B-CD4          0.997     0.124


In [23]:
.libPaths(c("/workspace/.Rlib", .libPaths()))
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## Scenario 4: B_priv, Mono_priv, CD8NK_shared (CD8+NK), IFN_global (all 5 CTs)
sim4 <- simulate_programs(scenario = 4L, N = 120L, G = 400L, signal_sd = 3,
                          sparsity = 0.05, noise_sd = 1, seed = 42)
fit4 <- fit_celltype_programs(sim4$x, max_factors = 10L)
fit4 <- canonicalize_programs(fit4)
cat("programs per CT:", vapply(fit4$loadings, ncol, integer(1L)), "\n")

## all-pairs PVA (rewritten): expect exactly 2 axes -> IFN_global + CD8NK_shared
fit4 <- integrate_programs(fit4, method = "pva", n_perm = 200L, seed = 1L)
print(integration_summary(fit4))

programs per CT: 2 1 2 2 2 


  method state    pair    cosine     p_value
1    pva  PV01 NK-Mono 0.9960766 0.004975124
2    pva  PV02  CD8-NK 0.9931839 0.004975124


In [24]:
## correlate PVA states with true activities
S <- fit4$integration$states
truth_z <- lapply(sim4$truth$activities, `[[`, "z")
cor_tab <- sapply(truth_z, function(z) apply(S, 2, function(s) cor(s, z, use = "pairwise.complete.obs")))
print(round(cor_tab, 3))

## traceability: top genes of each state vs true loadings (uses fixed w_a/w_b fields)
for (st in colnames(S)) {
  tr <- trace_state(fit4, state = st, top_n = 5)
  cat("\n", st, "top genes per CT:\n")
  print(lapply(tr, names))
}
## true top-5 IFN genes in B and CD8NK_shared genes in CD8
cat("true IFN_global top5 in B:   ", names(sort(-abs(sim4$truth$loadings$B$IFN_global))[1:5]), "\n")
cat("true CD8NK_shared top5 in CD8:", names(sort(-abs(sim4$truth$loadings$CD8$CD8NK_shared))[1:5]), "\n")

     B_priv Mono_priv CD8NK_shared IFN_global
PV01 -0.016     0.039        0.128      0.999
PV02  0.054    -0.031       -0.885      0.341



 PV01 top genes per CT:
$NK
[1] "g291" "g192" "g42"  "g103" "g106"

$Mono
[1] "g43"  "g41"  "g227" "g94"  "g10" 


 PV02 top genes per CT:
$CD8
[1] "g57"  "g154" "g213" "g185" "g337"

$NK
[1] "g137" "g49"  "g175" "g291" "g99" 



true IFN_global top5 in B:    g312 g193 g19 g102 g343 


true CD8NK_shared top5 in CD8: g57 g154 g213 g185 g337 


In [25]:
## Negative controls: PVA must find ZERO axes
## Scenario 1: all programs private
sim1 <- simulate_programs(scenario = 1L, N = 120L, G = 400L, signal_sd = 3,
                          sparsity = 0.05, noise_sd = 1, seed = 42)
fit1 <- canonicalize_programs(fit_celltype_programs(sim1$x, max_factors = 10L))
fit1 <- integrate_programs(fit1, method = "pva", n_perm = 200L, seed = 1L)
cat("Scenario 1 PVA axes:", length(fit1$integration$axes), "\n")
print(fit1$integration$note %||% integration_summary(fit1))

## Scenario 7: same genes in B/Mono, independent activities (scores-space null)
sim7 <- simulate_programs(scenario = 7L, N = 120L, G = 400L, signal_sd = 3,
                          sparsity = 0.05, noise_sd = 1, seed = 42)
fit7 <- canonicalize_programs(fit_celltype_programs(sim7$x, max_factors = 10L))
fit7 <- integrate_programs(fit7, method = "pva", n_perm = 200L, seed = 1L)
cat("Scenario 7 PVA axes:", length(fit7$integration$axes), "\n")
print(fit7$integration$note %||% integration_summary(fit7))

Scenario 1 PVA axes: 1 


  method state   pair    cosine    p_value
1    pva  PV01 B-Mono 0.1291627 0.02985075


Scenario 7 PVA axes: 0 


[1] "no shared directions above the permutation null"


In [26]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)  # reload after edits

rerun_pva <- function(scenario) {
  sim <- simulate_programs(scenario = scenario, N = 120L, G = 400L, signal_sd = 3,
                           sparsity = 0.05, noise_sd = 1, seed = 42)
  f <- canonicalize_programs(fit_celltype_programs(sim$x, max_factors = 10L))
  f <- integrate_programs(f, method = "pva", n_perm = 200L, seed = 1L)
  list(fit = f, sim = sim)
}

for (sc in c(4L, 1L, 7L, 5L)) {
  r <- rerun_pva(sc)
  ax <- r$fit$integration$axes
  cat(sprintf("Scenario %d: %d PVA axes", sc, length(ax)))
  if (length(ax) > 0L) {
    S <- r$fit$integration$states
    tz <- lapply(r$sim$truth$activities, `[[`, "z")
    best <- vapply(colnames(S), function(s) {
      cc <- sapply(tz, function(z) abs(cor(S[, s], z, use = "pairwise.complete.obs")))
      sprintf("%s->%s(|r|=%.2f)", s, names(which.max(cc)), max(cc))
    }, character(1L))
    cat(":", paste(best, collapse = ", "))
  }
  cat("\n")
}

Scenario 4: 2 PVA axes: PV01->IFN_global(|r|=1.00), PV02->CD8NK_shared(|r|=0.88)
Scenario 1: 0 PVA axes
Scenario 7: 0 PVA axes
Scenario 5: 1 PVA axes: PV01->Mono_distinct(|r|=0.93)


In [27]:
## ---- L1: shared-prior borrowing (outer EB loop) ----
## (a) global pooling on Scenario 4 full data
fit4_l1 <- fit_celltype_programs(sim4$x, max_factors = 10L, share_prior = "global", seed = 42)
l1 <- fit4_l1$borrowing$level1
cat("L1 global: iterations =", l1$n_iter, "(max 5)\n")
print(round(l1$moments, 4))
cat("program counts:", vapply(fit4_l1$loadings, ncol, integer(1L)), "\n")
rec <- sapply(names(sim4$truth$activities), function(pn) {
  a <- sim4$truth$activities[[pn]]
  max(sapply(a$cell_types, function(ct) max(abs(cor(fit4_l1$scores[[ct]], a$z)))))
})
cat("activity recovery (max |cor| per true program):\n"); print(round(rec, 3))

## (b) group pooling (lymphoid vs myeloid)
fit4_l1g <- fit_celltype_programs(sim4$x, max_factors = 10L, share_prior = "groups",
                                  prior_groups = list(lymph = c("B","CD4","CD8","NK"),
                                                      myeloid = "Mono"), seed = 42)
cat("L1 groups: iterations =", fit4_l1g$borrowing$level1$n_iter, "\n")
print(round(fit4_l1g$borrowing$level1$moments, 4))

## (c) small-N stress: NK downsampled to 30 obs; does global pooling hurt/help?
m <- sim4$x$matrices; m$NK <- m$NK[1:30, ]
x_small <- as_cell_program_data(m)
f_none <- fit_celltype_programs(x_small, max_factors = 10L, seed = 42)
f_glob <- fit_celltype_programs(x_small, max_factors = 10L, share_prior = "global", seed = 42)
z_ifn <- sim4$truth$activities$IFN_global$z[1:30]
z_sh  <- sim4$truth$activities$CD8NK_shared$z[1:30]
for (nm in c("f_none", "f_glob")) {
  f <- get(nm); sc <- f$scores$NK
  cat(sprintf("%-7s NK programs = %d | best|cor| IFN = %.3f, CD8NK = %.3f\n",
              nm, ncol(sc), max(abs(cor(sc, z_ifn))), max(abs(cor(sc, z_sh)))))
}

L1 global: iterations = 2 (max 5)


               
1 0.0492 2.7005
2 0.0492 2.7005


program counts: 2 1 2 2 2 


activity recovery (max |cor| per true program):


      B_priv    Mono_priv CD8NK_shared   IFN_global 
       0.996        0.998        0.997        0.998 


L1 groups: iterations = 2 


               
1 0.0506 2.5569
2 0.0441 3.2134
3 0.0506 2.5569
4 0.0441 3.2134


f_none  NK programs = 2 | best|cor| IFN = 0.997, CD8NK = 0.994
f_glob  NK programs = 2 | best|cor| IFN = 0.997, CD8NK = 0.994


In [28]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)  # reload after fix

## ---- L2: initialization borrowing ----
## Test 1 (negative control): cross-CT donor with DIFFERENT gene usage (Scenario 4
## shared programs use CT-specific genes). Borrowed factors must shrink to zero.
fit4_xct <- fit_celltype_programs(sim4$x, max_factors = 10L, seed = 42,
                                  init_from = list(NK = list(fit = fit4, cell_type = "CD8")))
rep1 <- fit4_xct$borrowing$level2$init_map$NK
cat("Test 1: borrowed", rep1$n_borrowed, "donor factors,", rep1$n_overlap_genes, "overlap genes\n")
sc <- fit4_xct$scores$NK
cat(sprintf("NK programs = %d | best|cor| IFN = %.3f, CD8NK = %.3f (expect ~unchanged)\n",
            ncol(sc), max(abs(cor(sc, sim4$truth$activities$IFN_global$z))),
            max(abs(cor(sc, sim4$truth$activities$CD8NK_shared$z)))))

## Test 2 (real use case): same-CT donor across cohorts, weak-signal target.
## Cohort B: NK only, N = 30, true NK loadings scaled x0.5, fresh activities.
w1 <- sim4$truth$loadings$NK$CD8NK_shared
w2 <- sim4$truth$loadings$NK$IFN_global
set.seed(99); N_b <- 30L
z1 <- rnorm(N_b); z2 <- rnorm(N_b)
Yb <- 0.5 * (z1 %*% t(w1) + z2 %*% t(w2)) + matrix(rnorm(N_b * 400L), N_b, 400L)
rownames(Yb) <- paste0("b", seq_len(N_b))
x_b <- as_cell_program_data(list(NK = Yb))

f_b_alone <- fit_celltype_programs(x_b, max_factors = 10L, seed = 42)
f_b_init  <- fit_celltype_programs(x_b, max_factors = 10L, seed = 42,
                                   init_from = list(NK = list(fit = fit4, cell_type = "NK")))
for (nm in c("f_b_alone", "f_b_init")) {
  f <- get(nm); sc <- f$scores$NK
  r1 <- if (ncol(sc) > 0) max(abs(cor(sc, z1))) else NA
  r2 <- if (ncol(sc) > 0) max(abs(cor(sc, z2))) else NA
  cat(sprintf("%-10s programs = %d | best|cor| CD8NK = %.3f, IFN = %.3f\n",
              nm, ncol(sc), r1, r2))
}
print(f_b_init$borrowing$level2$init_map$NK)

  --Estimate of factor 2 is numerically zero!
  --Estimate of factor 2 is numerically zero!
  --Estimate of factor 1 is numerically zero!


Test 1: borrowed 2 donor factors, 400 overlap genes


NK programs = 2 | best|cor| IFN = 0.998, CD8NK = 0.997 (expect ~unchanged)


f_b_alone  programs = 2 | best|cor| CD8NK = 0.982, IFN = 0.986
f_b_init   programs = 2 | best|cor| CD8NK = 0.982, IFN = 0.986


$donor_cell_type
[1] "NK"

$n_borrowed
[1] 2

$n_overlap_genes
[1] 400



In [29]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)  # reload after 0-factor fix

## Harder stress: N = 20, signal x0.35 (SNR ~ 0.37 per active gene)
set.seed(100); N_c <- 20L
z1c <- rnorm(N_c); z2c <- rnorm(N_c)
Yc <- 0.35 * (z1c %*% t(w1) + z2c %*% t(w2)) + matrix(rnorm(N_c * 400L), N_c, 400L)
rownames(Yc) <- paste0("c", seq_len(N_c))
x_c <- as_cell_program_data(list(NK = Yc))
f_c_alone <- fit_celltype_programs(x_c, max_factors = 10L, seed = 42)
f_c_init  <- fit_celltype_programs(x_c, max_factors = 10L, seed = 42,
                                   init_from = list(NK = list(fit = fit4, cell_type = "NK")))
for (nm in c("f_c_alone", "f_c_init")) {
  f <- get(nm); sc <- f$scores$NK
  r1 <- if (ncol(sc) > 0) max(abs(cor(sc, z1c))) else NA_real_
  r2 <- if (ncol(sc) > 0) max(abs(cor(sc, z2c))) else NA_real_
  cat(sprintf("%-10s programs = %d | best|cor| CD8NK = %.3f, IFN = %.3f\n",
              nm, ncol(sc), r1, r2))
}
cat("init fit pve:", round(f_c_init$fits$NK$pve, 3), "\n")
cat("alone fit pve:", round(f_c_alone$fits$NK$pve, 3), "\n")

f_c_alone  programs = 0 | best|cor| CD8NK = NA, IFN = NA
f_c_init   programs = 0 | best|cor| CD8NK = NA, IFN = NA


ERROR: Error in round(f_c_init$fits$NK$pve, 3): non-numeric argument to mathematical function


In [30]:
## L2 value scan: transition-zone regimes, 3 seeds each
sweep <- expand.grid(N = c(20L, 30L, 40L), scl = c(0.35, 0.5), seed = 1:3)
res <- lapply(seq_len(nrow(sweep)), function(i) {
  N_i <- sweep$N[i]; s_i <- sweep$scl[i]
  set.seed(sweep$seed[i])
  z1 <- rnorm(N_i); z2 <- rnorm(N_i)
  Y <- s_i * (z1 %*% t(w1) + z2 %*% t(w2)) + matrix(rnorm(N_i * 400L), N_i, 400L)
  rownames(Y) <- paste0("o", seq_len(N_i))
  xc <- as_cell_program_data(list(NK = Y))
  fa <- fit_celltype_programs(xc, max_factors = 10L, seed = 42)
  fi <- fit_celltype_programs(xc, max_factors = 10L, seed = 42,
                              init_from = list(NK = list(fit = fit4, cell_type = "NK")))
  rec <- function(f) {
    sc <- f$scores$NK
    if (ncol(sc) == 0L) return(c(NA, NA))
    c(max(abs(cor(sc, z1))), max(abs(cor(sc, z2))))
  }
  ra <- rec(fa); ri <- rec(fi)
  data.frame(N = N_i, scale = s_i, seed = sweep$seed[i],
             K_alone = ncol(fa$scores$NK), K_init = ncol(fi$scores$NK),
             alone_min = round(pmin(ra[1], ra[2]), 3), init_min = round(pmin(ri[1], ri[2]), 3))
})
sweep_res <- do.call(rbind, res)
print(sweep_res)
cat("\nmean min-|cor| by regime (both programs recovered):\n")
print(aggregate(cbind(alone_min, init_min) ~ N + scale, sweep_res, mean, na.rm = TRUE))

  --Estimate of factor 4 is numerically zero!
  --Estimate of factor 4 is numerically zero!


    N scale seed K_alone K_init alone_min init_min
1  20  0.35    1       1      1     0.321    0.290
2  30  0.35    1       2      2     0.951    0.947
3  40  0.35    1       2      2     0.946    0.945
4  20  0.50    1       2      2     0.970    0.967
5  30  0.50    1       2      2     0.979    0.978
6  40  0.50    1       2      2     0.974    0.974
7  20  0.35    2       2      2     0.962    0.962
8  30  0.35    2       2      2     0.985    0.984
9  40  0.35    2       2      2     0.968    0.968
10 20  0.50    2       2      2     0.983    0.981
11 30  0.50    2       2      2     0.993    0.992
12 40  0.50    2       2      2     0.984    0.984
13 20  0.35    3       1      1     0.204    0.202
14 30  0.35    3       1      2     0.146    0.931
15 40  0.35    3       2      2     0.926    0.925
16 20  0.50    3       2      2     0.963    0.964
17 30  0.50    3       2      2     0.967    0.970
18 40  0.50    3       2      2     0.966    0.966



mean min-|cor| by regime (both programs recovered):


   N scale alone_min  init_min
1 20  0.35 0.4956667 0.4846667
2 30  0.35 0.6940000 0.9540000
3 40  0.35 0.9466667 0.9460000
4 20  0.50 0.9720000 0.9706667
5 30  0.50 0.9796667 0.9800000
6 40  0.50 0.9746667 0.9746667


In [31]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## ---- L3 candidate A: block-structured joint EBMF, scores level ----
blk_s <- fit_joint_block(fit4, level = "scores", order = "shared_first", seed = 1L)
cat("== scores level, shared_first ==\n")
print(blk_s$summary)
if (!is.null(blk_s$states) && ncol(blk_s$states) > 0L) {
  cc <- sapply(lapply(sim4$truth$activities, `[[`, "z"),
               function(z) apply(blk_s$states, 2, function(s) cor(s, z, use = "pairwise.complete.obs")))
  cat("\nstate vs truth correlations:\n"); print(round(cc, 3))
}
cat("\nprivate pass factors per CT:", vapply(blk_s$private_fits, function(f) if (is.null(f)) NA else f$n_factors, integer(1L)), "\n")

Warning message in report.maxiter.reached(verbose.lvl):
“Maximum number of iterations reached.”


Warning message in log(tau):
“NaNs produced”


Warning message in handle_standard_errors(x, s):
“Nonpositive SEs have been replaced by small positive SEs.”


Warning message:
“block private pass failed for 'CD4': missing value where TRUE/FALSE needed”


Warning message in report.maxiter.reached(verbose.lvl):
“Maximum number of iterations reached.”


Warning message in report.maxiter.reached(verbose.lvl):
“Maximum number of iterations reached.”


== scores level, shared_first ==


  factor_id          class   entropy       mass_B     mass_CD4  mass_CD8
1      SF01         global 0.9703241 1.651320e-01 3.319036e-01 0.1676033
2      SF02 partial:CD8+NK 0.4307015 2.427379e-06 1.514076e-07 0.5014276
    mass_NK    mass_Mono
1 0.1686861 1.666750e-01
2 0.4985691 6.704829e-07



state vs truth correlations:
     B_priv Mono_priv CD8NK_shared IFN_global
SF01  0.021    -0.036       -0.132     -0.999
SF02  0.069    -0.049       -0.994     -0.039


ERROR: Error in vapply(blk_s$private_fits, function(f) if (is.null(f)) NA else f$n_factors, : values must be type 'integer',
 but FUN(X[[1]]) result is type 'double'


In [32]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## scores level (private pass now skipped by design)
blk_s <- fit_joint_block(fit4, level = "scores", order = "shared_first", seed = 1L)
cat("== scores / shared_first ==\n"); print(blk_s$summary[, c("factor_id", "class", "entropy")])

## expression level, both orders
blk_e <- fit_joint_block(fit4, level = "expression", order = "shared_first", seed = 1L)
cat("\n== expression / shared_first ==\n"); print(blk_e$summary)
if (!is.null(blk_e$states) && ncol(blk_e$states) > 0L) {
  cc <- sapply(lapply(sim4$truth$activities, `[[`, "z"),
               function(z) apply(blk_e$states, 2, function(s) cor(s, z, use = "pairwise.complete.obs")))
  cat("state vs truth:\n"); print(round(cc, 3))
}
cat("private pass factors:", vapply(blk_e$private_fits, function(f) if (is.null(f)) NA_integer_ else as.integer(f$n_factors), integer(1L)), "\n")

blk_e_pf <- fit_joint_block(fit4, level = "expression", order = "private_first", seed = 1L)
cat("\n== expression / private_first (ablation: expect ~no shared factors) ==\n")
print(blk_e_pf$summary)

== scores / shared_first ==


  factor_id          class   entropy
1      SF01         global 0.9703241
2      SF02 partial:CD8+NK 0.4307015



== expression / shared_first ==


  factor_id          class      entropy       mass_B     mass_CD4     mass_CD8
1      SF01         global 9.357714e-01 8.282065e-02 1.283820e-01 1.923939e-01
2      SF02 partial:CD8+NK 4.172020e-01 3.366490e-07 2.489837e-06 6.038708e-01
3      SF03   private:Mono 7.735854e-04 9.964358e-05 1.396792e-07 1.888442e-05
4      SF04      private:B 3.267687e-05 9.999964e-01 5.114033e-07 3.117912e-07
       mass_NK    mass_Mono
1 2.707694e-01 3.256340e-01
2 3.961255e-01 8.839565e-07
3 3.618636e-08 9.998813e-01
4 1.598616e-06 1.141052e-06


state vs truth:
     B_priv Mono_priv CD8NK_shared IFN_global
SF01 -0.020     0.036        0.132      0.999
SF02 -0.069     0.050        0.999      0.132
SF03  0.051    -0.998       -0.040     -0.033
SF04  0.996    -0.055       -0.056     -0.004


private pass factors: 0 0 0 0 0 



== expression / private_first (ablation: expect ~no shared factors) ==


[1] factor_id class     entropy  
<0 rows> (or 0-length row.names)


In [33]:
## ---- EV-BIDIFAC oracle validation (reference used in-session only) ----
source("/workspace/reference/EV_BIDIFAC_Functions.R")

set.seed(7)
N <- 60; G <- 40
cols <- list(1:40, 41:80, 81:120)
z123 <- rnorm(N); w123 <- rnorm(3 * G, 0, 2)   # global module
z12  <- rnorm(N); w12  <- rnorm(2 * G, 0, 2)   # partial {1,2}
z3   <- rnorm(N); w3   <- rnorm(G, 0, 2)       # private {3}
X <- z123 %*% t(w123)
X[, 1:80]   <- X[, 1:80] + z12 %*% t(w12)
X[, 81:120] <- X[, 81:120] + z3 %*% t(w3)
X <- X + matrix(rnorm(N * 3 * G, 0, 0.5), N, 3 * G)

## reference: modules = 7 non-empty column subsets
subs <- list(1L, 2L, 3L, c(1L, 2L), c(1L, 3L), c(2L, 3L), c(1L, 2L, 3L))
p.ind <- list(seq_len(N)); n.ind <- cols
p.ind.list <- lapply(seq_along(subs), function(k) seq_len(N))
n.ind.list <- lapply(subs, function(s) unlist(cols[s]))
invisible(capture.output(
  ref <- glob.bidi.o(X, p.ind, n.ind, p.ind.list, n.ind.list,
                     kappa = get_kappa(X), conv.thresh = 1e-8, max.iter = 300)
))

## mine
mine <- evbidifac_decompose(X, col_sets = cols, max_iter = 500L, warm_iter = 50L,
                            conv_tol = 1e-8)

## compare per module, matched by column-set label
ref_label <- vapply(n.ind.list, function(cc) paste(match(cc, unlist(cols)), collapse = ""), character(1L))
ref_label <- vapply(subs, function(s) paste(s, collapse = "+"), character(1L))
my_label <- mine$modules$col_sets
cat(sprintf("%-8s %10s %10s %12s %12s\n", "module", "||S_ref||", "||S_mine||", "cosine", "rel_diff"))
for (k in seq_along(subs)) {
  lab <- ref_label[k]
  Sr <- ref$S[[k]][, n.ind.list[[k]], drop = FALSE]
  m <- match(lab, my_label)
  Sm <- mine$S[[m]]$S
  cs <- sum(Sr * Sm) / (sqrt(sum(Sr^2)) * sqrt(sum(Sm^2)) + 1e-12)
  rd <- sqrt(sum((Sr - Sm)^2)) / (sqrt(sum(Sr^2)) + 1e-12)
  cat(sprintf("%-8s %10.3f %10.3f %12.4f %12.4f\n", lab, sqrt(sum(Sr^2)), sqrt(sum(Sm^2)), cs, rd))
}
cat("sigma2 ref :", round(ref$sigma2mat, 4), "\n")
cat("sigma2 mine:", round(mine$sigma2, 4), "\n")
cat("total recon error ref :", round(sqrt(sum((X - Reduce("+", ref$S))^2)) / sqrt(sum(X^2)), 4), "\n")
myfit <- matrix(0, N, 3 * G)
for (k in seq_along(mine$S)) myfit[mine$S[[k]]$rows, mine$S[[k]]$cols] <- myfit[mine$S[[k]]$rows, mine$S[[k]]$cols] + mine$S[[k]]$S
cat("total recon error mine:", round(sqrt(sum((X - myfit)^2)) / sqrt(sum(X^2)), 4), "\n")

module    ||S_ref|| ||S_mine||       cosine     rel_diff


1             0.000      0.000       0.0000       0.0000
2             0.000      0.000       0.0000       0.0000
3            97.139      0.000       0.0000       1.0000
1+2         159.333      0.000       0.0000       1.0000
1+3           0.000      0.000       0.0000       0.0000
2+3           0.000      0.000       0.0000       0.0000
1+2+3       152.644      0.000       0.0000       1.0000


sigma2 ref : 0.2512 0.259 0.2578 


sigma2 mine: 9.3164 8.3778 7.4738 


total recon error ref : 0.1689 


total recon error mine: 1 


In [34]:
## diagnose sigma2 estimation: my .evb_sigma2 vs reference get_sigma on block 1
B1 <- X[, cols[[1]]]
cat("block dims:", dim(B1), " mean(X^2) =", mean(B1^2), " true noise var = 0.25\n")
cat("ref get_sigma :", get_sigma(B1), "\n")
cat("my .evb_sigma2:", .evb_sigma2(B1), "\n")

## compare psi objective curves over the sigma2 grid
d <- svd(B1)$d; d <- d[d > 1e-3]
M <- max(dim(B1)); Nn <- min(dim(B1)); alpha <- Nn / M
kap_r <- get_kappa(B1); kap_m <- .evb_kappa(M, Nn)
cat("kappa ref:", kap_r, " mine:", kap_m, "\n")
s2_grid <- seq(0.05, mean(B1^2), length.out = 200)
psi_r <- sapply(s2_grid, function(s2) sum(psi(d^2 / (M * s2), alpha, kap_r, H = length(d))))
psi_m <- sapply(s2_grid, function(s2) sum(.evb_psi(d^2 / (M * s2), alpha, kap_m)))
cat("ref psi minimized at s2 =", s2_grid[which.min(psi_r)], "\n")
cat("my  psi minimized at s2 =", s2_grid[which.min(psi_m)], "\n")
cat("psi curve (ref) at s2=0.25, 1, 4, 8:", round(psi_r[which.min(abs(s2_grid-0.25))], 3), round(psi_r[which.min(abs(s2_grid-1))], 3), round(psi_r[which.min(abs(s2_grid-4))], 3), round(psi_r[which.min(abs(s2_grid-8))], 3), "\n")
cat("psi curve (mine) same points     :", round(psi_m[which.min(abs(s2_grid-0.25))], 3), round(psi_m[which.min(abs(s2_grid-1))], 3), round(psi_m[which.min(abs(s2_grid-4))], 3), round(psi_m[which.min(abs(s2_grid-8))], 3), "\n")

block dims: 60 40  mean(X^2) = 9.31644  true noise var = 0.25


ref get_sigma : 0.2511798 


my .evb_sigma2: 0.2595161 


kappa ref: 2.51569  mine: 2.515703 


ref psi minimized at s2 = 0.2362601 


my  psi minimized at s2 = 0.2362601 


psi curve (ref) at s2=0.25, 1, 4, 8: 68.218 90.794 135.27 159.485 


psi curve (mine) same points     : 68.218 90.794 135.27 159.485 


In [35]:
## instrument evbidifac_decompose internals step by step
row_sets <- list(seq_len(N)); col_sets <- cols
modules <- list()
for (rsub in .powerset(1L)) for (csub in .powerset(1:3)) modules[[length(modules)+1L]] <- list(rows = rsub, cols = csub)
K <- length(modules)
mod_rows <- lapply(modules, function(m) unlist(row_sets[m$rows]))
mod_cols <- lapply(modules, function(m) unlist(col_sets[m$cols]))

sigma2 <- matrix(1, 1, 3)
for (j in 1:3) sigma2[1, j] <- .evb_sigma2(X[, col_sets[[j]]])
cat("initial sigma2:", sigma2, "\n")
Xs <- X
for (j in 1:3) Xs[, col_sets[[j]]] <- Xs[, col_sets[[j]]] / sqrt(sigma2[1, j])

S <- lapply(seq_len(K), function(k) matrix(0, length(mod_rows[[k]]), length(mod_cols[[k]])))
Fitted <- matrix(0, N, 3 * G)

## one warm iteration manually
delta <- 0
for (k in seq_len(K)) {
  rr <- mod_rows[[k]]; cc <- mod_cols[[k]]
  R_sub <- Xs[rr, cc, drop = FALSE] - Fitted[rr, cc, drop = FALSE] + S[[k]]
  newS <- .nn_shrink(R_sub)
  delta <- delta + sum((newS - S[[k]])^2)
  Fitted[rr, cc] <- Fitted[rr, cc] + (newS - S[[k]])
  S[[k]] <- newS
  cat(sprintf("module %d cols=%s: max|d_new| frob = %.3f\n", k, paste(modules[[k]]$cols, collapse="+"), sqrt(sum(newS^2))))
}
cat("warm iter 1 delta:", delta, "\n")

initial sigma2: 0.2595161 0.2666109 0.2526711 


module 1 cols=1: max|d_new| frob = 270.233
module 2 cols=2: max|d_new| frob = 250.925
module 3 cols=3: max|d_new| frob = 242.509
module 4 cols=1+2: max|d_new| frob = 4.531
module 5 cols=1+3: max|d_new| frob = 1.672
module 6 cols=2+3: max|d_new| frob = 0.711
module 7 cols=1+2+3: max|d_new| frob = 1.287


warm iter 1 delta: 194825.2 


In [36]:
mine2 <- evbidifac_decompose(X, col_sets = cols, max_iter = 500L, warm_iter = 50L,
                             conv_tol = 1e-8, verbose = FALSE)
cat("n_iter:", mine2$n_iter, " converged:", mine2$converged, "\n")
cat("sigma2:", mine2$sigma2, "\n")
print(mine2$modules)
cat("str of first S entry:\n"); str(mine2$S[[3]])

n_iter: 50 2  converged: TRUE 


sigma2: 9.31644 8.377761 7.473791 


  module_id row_sets col_sets n_col_sets       class rank frob
1       M01        1        1          1   private:1    0    0
2       M02        1        2          1   private:2    0    0
3       M03        1        3          1   private:3    0    0
4       M04        1      1+2          2 partial:1+2    0    0
5       M05        1      1+3          2 partial:1+3    0    0
6       M06        1      2+3          2 partial:2+3    0    0
7       M07        1    1+2+3          3      global    0    0


str of first S entry:


List of 3
 $ rows: int [1:60] 1 2 3 4 5 6 7 8 9 10 ...
 $ cols: int [1:40] 81 82 83 84 85 86 87 88 89 90 ...
 $ S   : num [1:60, 1:40] 0 0 0 0 0 0 0 0 0 0 ...


In [37]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)  # reload after sigma2 fix

mine <- evbidifac_decompose(X, col_sets = cols, max_iter = 500L, warm_iter = 50L,
                            conv_tol = 1e-8)
cat("n_iter:", mine$n_iter, " converged:", mine$converged, "\n")
cat("sigma2 mine:", round(mine$sigma2, 4), " (ref: 0.2512 0.2590 0.2578; true 0.25)\n\n")

cat(sprintf("%-8s %10s %10s %12s %12s\n", "module", "||S_ref||", "||S_mine||", "cosine", "rel_diff"))
for (k in seq_along(subs)) {
  lab <- ref_label[k]
  Sr <- ref$S[[k]][, n.ind.list[[k]], drop = FALSE]
  m <- match(lab, mine$modules$col_sets)
  Sm <- mine$S[[m]]$S
  cs <- sum(Sr * Sm) / (sqrt(sum(Sr^2)) * sqrt(sum(Sm^2)) + 1e-12)
  rd <- sqrt(sum((Sr - Sm)^2)) / (sqrt(sum(Sr^2)) + 1e-12)
  cat(sprintf("%-8s %10.3f %10.3f %12.4f %12.4f\n", lab, sqrt(sum(Sr^2)), sqrt(sum(Sm^2)), cs, rd))
}
myfit <- matrix(0, N, 3 * G)
for (k in seq_along(mine$S)) myfit[mine$S[[k]]$rows, mine$S[[k]]$cols] <- myfit[mine$S[[k]]$rows, mine$S[[k]]$cols] + mine$S[[k]]$S
cat("\ntotal recon error ref :", round(sqrt(sum((X - Reduce("+", ref$S))^2)) / sqrt(sum(X^2)), 4), "\n")
cat("total recon error mine:", round(sqrt(sum((X - myfit)^2)) / sqrt(sum(X^2)), 4), "\n")

n_iter: 50 455  converged: TRUE 


sigma2 mine: 0.2595 0.2666 0.2527  (ref: 0.2512 0.2590 0.2578; true 0.25)



module    ||S_ref|| ||S_mine||       cosine     rel_diff


1             0.000      0.000       0.0000       0.0000
2             0.000      0.000       0.0000       0.0000
3            97.139     96.951       0.9993       0.0370
1+2         159.333    192.139       0.8301       0.6725
1+3           0.000     98.048       0.0000 98048484201305.1875
2+3           0.000      0.000       0.0000       0.0000
1+2+3       152.644      0.000       0.0000       1.0000



total recon error ref : 0.1689 


total recon error mine: 0.1675 


In [38]:
## hypothesis: warm phase must converge (like reference) to reach the right basin
mine_w <- evbidifac_decompose(X, col_sets = cols, max_iter = 500L, warm_iter = 500L,
                              conv_tol = 1e-8)
cat("n_iter:", mine_w$n_iter, "\n")
cat(sprintf("%-8s %10s %10s %12s\n", "module", "||S_ref||", "||S_mine||", "cosine"))
for (k in seq_along(subs)) {
  lab <- ref_label[k]
  Sr <- ref$S[[k]][, n.ind.list[[k]], drop = FALSE]
  m <- match(lab, mine_w$modules$col_sets)
  Sm <- mine_w$S[[m]]$S
  cs <- sum(Sr * Sm) / (sqrt(sum(Sr^2)) * sqrt(sum(Sm^2)) + 1e-12)
  cat(sprintf("%-8s %10.3f %10.3f %12.4f\n", lab, sqrt(sum(Sr^2)), sqrt(sum(Sm^2)), cs))
}

n_iter: 195 7 


module    ||S_ref|| ||S_mine||       cosine


1             0.000      0.000       0.0000
2             0.000      0.000       0.0000
3            97.139     97.170       1.0000
1+2         159.333    159.265       1.0000
1+3           0.000      0.000       0.0000
2+3           0.000      0.000       0.0000
1+2+3       152.644    152.499       1.0000


In [39]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## ---- EV-BIDIFAC on Scenario 4, scores level ----
t0 <- Sys.time()
evb_s <- fit_joint_evbidifac(fit4, level = "scores")
t_s <- difftime(Sys.time(), t0, units = "secs")
cat("scores-level runtime:", round(t_s, 1), "s\n")
print(evb_s$summary)

## correlate states with truth
keep_states <- colnames(evb_s$states)
if (length(keep_states) > 0L) {
  cc <- sapply(lapply(sim4$truth$activities, `[[`, "z"), function(z)
    apply(evb_s$states, 2, function(s) cor(s, z, use = "pairwise.complete.obs")))
  cc <- cc[, colSums(is.na(cc)) == 0, drop = FALSE]
  cat("\nstate vs truth:\n"); print(round(cc, 3))
}

Warning message in (function (..., deparse.level = 1) :
“number of rows of result is not a multiple of vector length (arg 1)”


scores-level runtime: 0.1 s


        state module       class  variance
M13_r1 M13_r1    M13 partial:3+4 0.1960834
M31_r1 M31_r1    M31      global 1.8334525



state vs truth:
       B_priv Mono_priv CD8NK_shared IFN_global
M13_r1 -0.068     0.048        0.992      0.021
M31_r1  0.029    -0.046       -0.165     -0.999


In [40]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## scores level re-run (confirm warning gone, check program_weights embedding)
evb_s <- fit_joint_evbidifac(fit4, level = "scores")
print(evb_s$summary)
cat("\nprogram_weights (rows = stacked program columns):\n")
print(round(evb_s$program_weights, 3))

## ---- expression level (full 31-module enumeration on 120 x 2000) ----
t0 <- Sys.time()
evb_e <- fit_joint_evbidifac(fit4, level = "expression")
t_e <- difftime(Sys.time(), t0, units = "mins")
cat("\nexpression-level runtime:", round(t_e, 2), "min\n")
print(evb_e$summary)
if (ncol(evb_e$states) > 0L) {
  cc <- sapply(lapply(sim4$truth$activities, `[[`, "z"), function(z)
    apply(evb_e$states, 2, function(s) cor(s, z, use = "pairwise.complete.obs")))
  cat("\nstate vs truth:\n"); print(round(cc, 3))
}

        state module       class  variance
M13_r1 M13_r1    M13 partial:3+4 0.1960834
M31_r1 M31_r1    M31      global 1.8334525



program_weights (rows = stacked program columns):


                 M13_r1 M31_r1
B__B__F001        0.000 -0.007
B__B__F002        0.000 -0.406
CD4__CD4__F001    0.000  0.575
CD8__CD8__F001   -0.708  0.054
CD8__CD8__F002    0.004  0.407
NK__NK__F001      0.015  0.407
NK__NK__F002     -0.706  0.063
Mono__Mono__F001  0.000  0.018
Mono__Mono__F002  0.000 -0.407



expression-level runtime: 6.59 min


        state module           class variance
M01_r1 M01_r1    M01       private:1 15002.06
M05_r1 M05_r1    M05       private:5 30701.98
M13_r1 M13_r1    M13     partial:3+4 40534.05
M29_r1 M29_r1    M29 partial:1+3+4+5 11426.94
M30_r1 M30_r1    M30 partial:2+3+4+5 89114.96



state vs truth:
       B_priv Mono_priv CD8NK_shared IFN_global
M01_r1 -0.950     0.043        0.015     -0.282
M05_r1 -0.050     0.998        0.037      0.012
M13_r1 -0.064     0.048        0.997      0.071
M29_r1  0.480    -0.049       -0.143     -0.873
M30_r1 -0.028    -0.037       -0.122     -0.998


In [41]:
## convergence diagnostics for the expression-level run
cat("expression-level n_iter:", evb_e$decomposition$n_iter,
    " converged:", evb_e$decomposition$converged, "\n")
print(evb_e$decomposition$modules[evb_e$decomposition$modules$frob > 1e-6,
                                  c("module_id", "col_sets", "class", "rank", "frob")])

## save the stacked expression matrix for the reference-oracle background run
st_e <- .build_stacked(fit4, "expression")
saveRDS(st_e$X, "/workspace/evb_oracle_input.rds")
cat("\nsaved stacked matrix:", nrow(st_e$X), "x", ncol(st_e$X), "\n")

expression-level n_iter: 258 200  converged: FALSE 


   module_id col_sets           class rank     frob
1        M01        1       private:1    1 122.4829
5        M05        5       private:5    1 175.2198
13       M13      3+4     partial:3+4    1 201.3307
29       M29  1+3+4+5 partial:1+3+4+5    1 106.8969
30       M30  2+3+4+5 partial:2+3+4+5    1 298.5213



saved stacked matrix: 120 x 2000 


In [42]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## ---- L3 mini-race harness (scores-level both candidates + block expression) ----
race_states <- function(scenario, seed = 42L) {
  sim <- simulate_programs(scenario = scenario, N = 120L, G = 400L, signal_sd = 3,
                           sparsity = 0.05, noise_sd = 1, seed = seed)
  fit <- canonicalize_programs(fit_celltype_programs(sim$x, max_factors = 10L))
  tz <- lapply(sim$truth$activities, `[[`, "z")

  cands <- list(
    block_S = fit_joint_block(fit, level = "scores", order = "shared_first", seed = 1L),
    block_E = fit_joint_block(fit, level = "expression", order = "shared_first", seed = 1L),
    evb_S   = fit_joint_evbidifac(fit, level = "scores")
  )
  rows <- list()
  for (nm in names(cands)) {
    cd <- cands[[nm]]
    summ <- cd$summary
    ids <- if ("factor_id" %in% names(summ)) summ$factor_id else summ$state
    cls <- summ$class
    if (length(ids) == 0L || is.null(cd$states) || ncol(cd$states) == 0L) {
      rows[[length(rows) + 1L]] <- data.frame(scenario = scenario, candidate = nm,
        state = "(none)", class = "-", best_truth = "-", best_cor = NA_real_)
      next
    }
    for (j in seq_along(ids)) {
      s <- cd$states[, ids[j]]
      cc <- sapply(tz, function(z) abs(cor(s, z, use = "pairwise.complete.obs")))
      rows[[length(rows) + 1L]] <- data.frame(
        scenario = scenario, candidate = nm, state = ids[j], class = cls[j],
        best_truth = names(which.max(cc)), best_cor = round(max(cc), 3))
    }
  }
  do.call(rbind, rows)
}

race_all <- lapply(c(1L, 2L, 3L, 4L, 5L, 6L, 7L), race_states)
race_df <- do.call(rbind, race_all)
print(race_df, row.names = FALSE)

 scenario candidate  state          class    best_truth best_cor
        1   block_S (none)              -             -       NA
        1   block_E   SF01    private:CD8        CD8_P1    0.997
        1   block_E   SF02      private:B          B_P2    0.998
        1   block_E   SF03    private:CD8        CD8_P2    0.998
        1   block_E   SF04     private:NK         NK_P2    0.997
        1   block_E   SF05    private:CD4        CD4_P1    0.998
        1   block_E   SF06     private:NK         NK_P1    0.997
        1   block_E   SF07   private:Mono       Mono_P2    0.998
        1   block_E   SF08    private:CD4        CD4_P2    0.996
        1   block_E   SF09      private:B          B_P1    0.996
        1   block_E   SF10   private:Mono       Mono_P1    0.996
        1     evb_S (none)              -             -       NA
        2   block_S   SF01         global      GLOBAL_1    0.999
        2   block_E   SF01         global      GLOBAL_1    0.999
        2   block_E   SF0

In [43]:
print(race_df[race_df$scenario %in% c(2L, 3L, 4L, 5L), ], row.names = FALSE)

 scenario candidate  state          class    best_truth best_cor
        2   block_S   SF01         global      GLOBAL_1    0.999
        2   block_E   SF01         global      GLOBAL_1    0.999
        2   block_E   SF02    private:CD4        CD4_P1    0.997
        2   block_E   SF03      private:B          B_P1    0.998
        2   block_E   SF04     private:NK         NK_P1    0.997
        2   block_E   SF05    private:CD8        CD8_P1    0.995
        2   block_E   SF06   private:Mono       Mono_P1    0.995
        2     evb_S M31_r1         global      GLOBAL_1    0.998
        3   block_S   SF01 partial:CD8+NK  CD8NK_shared    0.998
        3   block_E   SF01   private:Mono       Mono_P1    0.999
        3   block_E   SF02     private:NK         NK_P1    0.999
        3   block_E   SF03 partial:CD8+NK  CD8NK_shared    0.998
        3   block_E   SF04    private:CD4        CD4_P1    0.997
        3   block_E   SF05      private:B          B_P1    0.997
        3   block_E   SF0

In [44]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)

## stability smoke test: n_boot = 10, timed (nothing else running now)
t0 <- Sys.time()
stab4 <- assess_program_stability(fit4, n_boot = 10L, seed = 7L, progress = FALSE)
cat("stability runtime (10 boots):", round(difftime(Sys.time(), t0, units = "secs"), 1), "s\n")
print(stab4$stability)

stability runtime (10 boots): 1184.2 s


$per_program
  program_id cell_type recovery_freq median_score_cor median_loading_cos
1    B__F001         B           1.1        0.9998874          0.9952770
2    B__F002         B           0.9        0.9996657          0.9962953
3  CD4__F001       CD4           1.0        0.9998598          0.9971367
4  CD8__F001       CD8           1.9        0.9999086          0.9973820
5  CD8__F002       CD8           0.1        0.9998862          0.9982018
6   NK__F001        NK           2.0        0.9999200          0.9988533
7   NK__F002        NK           0.0               NA                 NA
8 Mono__F001      Mono           1.1        0.9998467          0.9996856
9 Mono__F002      Mono           0.0               NA                 NA
  sign_consistency top_gene_jaccard
1                1        0.6666667
2                1        0.9047619
3                1        0.8181818
4                1        0.8181818
5                1        0.9047619
6                1        0.9047619
7    

In [45]:
## profile ONE bootstrap iteration: fit timing per CT + match table for NK
set.seed(7)
mats_b <- lapply(fit4$input$matrices, function(Y) {
  idx <- sample.int(nrow(Y), replace = TRUE)
  Yb <- Y[idx, , drop = FALSE]
  rownames(Yb) <- paste0(rownames(Y)[idx], "__b1_", seq_len(nrow(Yb)))
  Yb
})
x_b <- as_cell_program_data(mats_b)
params <- fit4$provenance$parameters
for (ct in c("B", "NK")) {
  t0 <- Sys.time()
  f1 <- flashier::flash(x_b$matrices[[ct]], ebnm_fn = list(ebnm::ebnm_normal, ebnm::ebnm_point_normal),
                        var_type = 2L, greedy_Kmax = 10L, backfit = TRUE, nullcheck = TRUE, verbose = 0L)
  cat(ct, ": flash", round(difftime(Sys.time(), t0, units = "secs"), 1), "s, factors:", f1$n_factors, "\n")
}
## compare: same fits on the ORIGINAL (non-resampled) data
for (ct in c("B", "NK")) {
  t0 <- Sys.time()
  f1 <- flashier::flash(fit4$input$matrices[[ct]], ebnm_fn = list(ebnm::ebnm_normal, ebnm::ebnm_point_normal),
                        var_type = 2L, greedy_Kmax = 10L, backfit = TRUE, nullcheck = TRUE, verbose = 0L)
  cat(ct, "original:", round(difftime(Sys.time(), t0, units = "secs"), 1), "s, factors:", f1$n_factors, "\n")
}

B : flash 9.3 s, factors: 10 
NK : flash 8 s, factors: 10 


B original: 0.2 s, factors: 2 
NK original: 0.4 s, factors: 2 


In [46]:
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)  # reload greedy_match fix

## sanity: match table must be 1:1 now
mp <- match_programs(fit4, fit_b, "NK", method = "subspace", sim_threshold = 0.5)
cat("NK matches (fixed):\n"); print(mp$matches)

## full stability, 20 boots
stab4 <- assess_program_stability(fit4, n_boot = 20L, seed = 7L, progress = FALSE)
print(stab4$stability$per_program)

## match_programs: same data, different seed -> all programs match
fit4_b <- canonicalize_programs(fit_celltype_programs(sim4$x, max_factors = 10L, seed = 99L))
for (ct in c("B", "CD8")) {
  mm <- match_programs(fit4, fit4_b, cell_type = ct, method = "subspace", space = "loadings")
  cat("\n", ct, "matches:\n"); print(mm$matches)
}

NK matches (fixed):


  program_a program_b    cosine
1  NK__F001  NK__F001 0.9999391
2  NK__F002  NK__F002 0.9998659


  program_id cell_type recovery_freq median_score_cor median_loading_cos
1    B__F001         B             1        0.9999925          0.9998396
2    B__F002         B             1        0.9999616          0.9998483
3  CD4__F001       CD4             1        0.9999849          0.9998268
4  CD8__F001       CD8             1        0.9999889          0.9999309
5  CD8__F002       CD8             1        0.9999941          0.9998317
6   NK__F001        NK             1        0.9999862          0.9999533
7   NK__F002        NK             1        0.9999528          0.9998685
8 Mono__F001      Mono             1        0.9999361          0.9998969
9 Mono__F002      Mono             1        0.9999894          0.9999624
  sign_consistency top_gene_jaccard
1                1        0.7391304
2                1        0.9047619
3                1        0.9047619
4                1        1.0000000
5                1        0.9047619
6                1        1.0000000
7                1


 B matches:
  program_a program_b cosine
1   B__F001   B__F001      1
2   B__F002   B__F002      1

 CD8 matches:
  program_a program_b cosine
1 CD8__F001 CD8__F001      1
2 CD8__F002 CD8__F002      1


In [47]:
## ---- L3 race summary tables ----
## state-level table: fast candidates from race_df + evb-E Scenario 4 results
evb_e_s4 <- data.frame(
  scenario = 4L, candidate = "evb_E",
  state = c("M01_r1", "M05_r1", "M13_r1", "M31_r1"),
  class = c("private:B", "private:Mono", "partial:CD8+NK", "global"),
  best_truth = c("B_priv", "Mono_priv", "CD8NK_shared", "IFN_global"),
  best_cor = c(0.950, 0.998, 0.997, NA_real_)  # M31 from converged max_iter=1000 run
)
## converged run (max_iter=1000): global module consolidated; correlate its state
dec_long <- readRDS("/workspace/evb_mine_s4_maxiter1000.rds")$dec
gmod <- dec_long$modules[dec_long$modules$col_sets == "1+2+3+4+5", ]
cat("converged evb-E global module frob:", round(gmod$frob, 1), "\n")

race_table <- rbind(race_df, evb_e_s4)
write.csv(race_table, "/mnt/results/l3_race_state_table.csv", row.names = FALSE)

## criterion-level scorecard (decision table)
scorecard <- data.frame(
  criterion = c("shared-state recovery (S2/S4/S6)",
                "false sharing: S1 (all private)",
                "false sharing: S5 (correlated distinct)",
                "false sharing: S7 (same genes, indep. activity)",
                "private recovery (S1/S4)",
                "partial-sharing membership (S3/S4)",
                "runtime on S4 (approx.)",
                "convergence sensitivity"),
  block_S = c("0.994-0.999 all shared states",
              "none (clean)",
              "flags B+Mono (0.929) - inherent scores-space confound",
              "none (clean)",
              "n/a (private = stage-1 programs by construction)",
              "exact (CD8+NK)",
              "~1 s",
              "robust (greedy+backfit EBMF)"),
  block_E = c("0.996-0.999 all shared states",
              "none; 10/10 privates correct",
              "ROBUST: separates by gene usage (both private, 0.995-0.997)",
              "none; 5/5 privates correct",
              "10/10 (S1), 2/2 (S4) at 0.996-0.999",
              "exact (CD8+NK)",
              "~2 s",
              "robust"),
  evb_S = c("0.992-0.999 all shared states",
            "none (clean)",
            "flags B+Mono (0.925) - inherent scores-space confound",
            "none (clean)",
            "n/a (private modules do not survive at scores level)",
            "exact (CD8+NK)",
            "~5 s",
            "robust at scores level"),
  evb_E = c("0.95-0.998 ONLY with 4-5x iteration budget; global module SPLITS into\noverlapping partials at default budget (reproduced in authors' reference code)",
            "pending (background)",
            "pending (background)",
            "pending (background)",
            "recovered (B, Mono privates correct)",
            "exact (CD8+NK)",
            "6.6 min (default, unconverged) / 75 min (converged)",
            "SENSITIVE: needs max_iter >= 1000 at expression level"),
  stringsAsFactors = FALSE
)
write.csv(scorecard, "/mnt/results/l3_race_scorecard.csv", row.names = FALSE)
print(scorecard[, c("criterion", "block_E", "evb_E")], right = FALSE)

converged evb-E global module frob: 334.4 


  criterion                                      
1 shared-state recovery (S2/S4/S6)               
2 false sharing: S1 (all private)                
3 false sharing: S5 (correlated distinct)        
4 false sharing: S7 (same genes, indep. activity)
5 private recovery (S1/S4)                       
6 partial-sharing membership (S3/S4)             
7 runtime on S4 (approx.)                        
8 convergence sensitivity                        
  block_E                                                    
1 0.996-0.999 all shared states                              
2 none; 10/10 privates correct                               
3 ROBUST: separates by gene usage (both private, 0.995-0.997)
4 none; 5/5 privates correct                                 
5 10/10 (S1), 2/2 (S4) at 0.996-0.999                        
6 exact (CD8+NK)                                             
7 ~2 s                                                       
8 robust                                              

In [48]:
import cellxgene_census, pandas as pd

CENSUS_VERSION = "2025-11-08"  # pin for reproducibility
COMBAT_COLLECTION = "8f126edf-5405-4731-8374-b5ce11f53e82"

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    ds = census["census_info"]["datasets"].read().concat().to_pandas()
    combat_ds = ds[ds.collection_id == COMBAT_COLLECTION]
    print("datasets table columns:", list(ds.columns))
    keep_cols = [c for c in ["dataset_id", "dataset_title", "dataset_total_cell_count"] if c in ds.columns]
    print("\nCOMBAT datasets in collection:")
    print(combat_ds[keep_cols].to_string(index=False))

datasets table columns: ['soma_joinid', 'citation', 'collection_id', 'collection_name', 'collection_doi', 'collection_doi_label', 'dataset_id', 'dataset_version_id', 'dataset_title', 'dataset_h5ad_path', 'dataset_total_cell_count']

COMBAT datasets in collection:
                          dataset_id                                                                                dataset_title  dataset_total_cell_count
ebc2e1ff-c8f9-466a-acf4-9d291afaf8b3 COMBAT project: single cell gene expression data from COVID-19, sepsis and flu patient PBMCs                    836148


In [49]:
COMBAT_DS = "ebc2e1ff-c8f9-466a-acf4-9d291afaf8b3"

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    obs = (census["census_data"]["homo_sapiens"].obs.read(
        value_filter=f"dataset_id == '{COMBAT_DS}' and is_primary_data == True")
        .concat().to_pandas())

print("obs shape:", obs.shape)
print("obs columns:", list(obs.columns))
for col in ["disease", "cell_type", "tissue_general", "assay", "suspension_type"]:
    if col in obs.columns:
        print(f"\n== {col} ==")
        print(obs[col].value_counts().to_string())
print("\ndonors:", obs.donor_id.nunique())

obs shape: (836148, 28)
obs columns: ['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars']

== disease ==
disease
COVID-19                                                                                  724710
normal                                                                                     92205
influenza                                                                                  19233
lymphangioleiomyomatosis                                                                   

In [50]:
for col in ["disease", "cell_type", "assay", "sex", "development_stage", "self_reported_ethnicity"]:
    vc = obs[col].value_counts()
    vc = vc[vc > 0]
    print(f"== {col} ({len(vc)} observed) ==")
    print(vc.to_string())
    print()

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    ds = census["census_info"]["datasets"].read().concat().to_pandas()
row = ds[ds.dataset_id == COMBAT_DS].iloc[0]
print("h5ad path:", row.dataset_h5ad_path)
print("citation:", row.citation)

== disease (3 observed) ==
disease
COVID-19     724710
normal        92205
influenza     19233

== cell_type (17 observed) ==
cell_type
CD4-positive, alpha-beta T cell            269661
classical monocyte                         213697
CD8-positive, alpha-beta T cell            106025
natural killer cell                         70299
blood cell                                  52444
B cell                                      44256
non-classical monocyte                      33172
gamma-delta T cell                           9580
dendritic cell                               8959
plasmablast                                  8596
double-positive, alpha-beta thymocyte        6561
double negative T regulatory cell            4557
mucosal invariant T cell                     4541
hematopoietic stem cell                      1868
megakaryocyte-erythroid progenitor cell      1563
enucleated reticulocyte                       304
mast cell                                      65

== assay (1 o

h5ad path: ebc2e1ff-c8f9-466a-acf4-9d291afaf8b3.h5ad
citation: Publication: https://doi.org/10.1016/j.cell.2022.01.012 Dataset Version: https://datasets.cellxgene.cziscience.com/687c09ff-731a-4e3d-ac07-4c29c33a6338.h5ad curated and distributed by CZ CELLxGENE Discover in Collection: https://cellxgene.cziscience.com/collections/8f126edf-5405-4731-8374-b5ce11f53e82


In [51]:
import cellxgene_census, pandas as pd

COLLECTIONS = {
    "stephenson": "ddfad306-714d-4cc0-9985-d9072820c530",
    "onek1k":     "dde06e0f-ab3b-46be-96a2-a8082383c4a1",
}
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    ds = census["census_info"]["datasets"].read().concat().to_pandas()

for name, cid in COLLECTIONS.items():
    sub = ds[ds.collection_id == cid]
    print(f"== {name}: {len(sub)} datasets ==")
    print(sub[["dataset_id","dataset_title","dataset_total_cell_count","dataset_h5ad_path"]].to_string(index=False), "\n")

# HLCA + Smillie: find collections by name
allc = ds[["collection_id","collection_name"]].drop_duplicates()
for pat in ["lung cell atlas", "Smillie", "colon", "ulcerative"]:
    hits = allc[allc.collection_name.str.contains(pat, case=False, na=False)]
    print(f"-- collections matching '{pat}':")
    print(hits.to_string(index=False), "\n")

== stephenson: 1 datasets ==
                          dataset_id                                                       dataset_title  dataset_total_cell_count                         dataset_h5ad_path
c7775e88-49bf-4ba2-a03b-93f00447c958 Single-cell multi-omics analysis of the immune response in COVID-19                    647366 c7775e88-49bf-4ba2-a03b-93f00447c958.h5ad 

== onek1k: 1 datasets ==
                          dataset_id                                                                                dataset_title  dataset_total_cell_count                         dataset_h5ad_path
3faad104-2ab8-4434-816d-474d8d2641db Single-cell eQTL mapping identifies cell type specific genetic control of autoimmune disease                   1248980 3faad104-2ab8-4434-816d-474d8d2641db.h5ad 

-- collections matching 'lung cell atlas':
                       collection_id                                                                                                            collection_na

In [52]:
lung = allc[allc.collection_name.str.contains("lung", case=False, na=False)]
print(lung.to_string(index=False))
print()
sm = ds[ds.collection_id == "33d19f34-87f5-455b-8ca5-9023a2e5453d"]
print("Smillie datasets:")
print(sm[["dataset_id","dataset_title","dataset_total_cell_count"]].to_string(index=False))

                       collection_id                                                                                                                                      collection_name
2d2e2acd-dade-489f-a2da-6c11aa654028                           A human fetal lung cell atlas uncovers proximal-distal gradients of differentiation and key regulators of epithelial fates
29f92179-ca10-4309-a32b-d383d80347c1                 Longitudinal profiling of respiratory and systemic immune responses reveals myeloid cell-driven lung inflammation in severe COVID-19
c1241244-b22d-483d-875b-75699efb9f3c                                                           A spatially resolved atlas of the human lung characterizes a gland-associated immune niche
28e9d721-6816-48a2-8d0b-43bf0b0c0ebc                                                     Single cell transcriptomic profiling identifies molecular phenotypes of newborn human lung cells
92db8a7e-8761-4317-873f-15606c016ce5 A novel human fetal lung-derived 

In [53]:
hlca = ds[ds.collection_id == "6f6d381a-7701-4781-935c-db10d30de293"]
print("HLCA datasets:")
print(hlca[["dataset_id","dataset_title","dataset_total_cell_count","dataset_h5ad_path"]].to_string(index=False))

HLCA datasets:
                          dataset_id                                                           dataset_title  dataset_total_cell_count                         dataset_h5ad_path
066943a2-fdac-4b29-b348-40cede398e4e An integrated cell atlas of the human lung in health and disease (core)                    584944 066943a2-fdac-4b29-b348-40cede398e4e.h5ad
9f222629-9e39-47d0-b83f-e08d610c7479 An integrated cell atlas of the human lung in health and disease (full)                   2282447 9f222629-9e39-47d0-b83f-e08d610c7479.h5ad


In [54]:
.libPaths(c("/workspace/.Rlib", .libPaths()))
library(pkgload)
pkgload::load_all("/workspace/cellprograms", quiet = TRUE)
source("/workspace/benchmarks/R/metrics.R")
source("/workspace/benchmarks/R/baselines.R")

BASE <- "/mnt/shared-workspace/shared/combat"
cts <- c("CD4","CD8","B","NK","cMono","ncMono","DC","PB","GDT","DP")
mats <- lapply(cts, function(ct) {
  as.matrix(read.csv(file.path(BASE, "mat", paste0("ct_", ct, ".csv")), row.names = 1))
})
names(mats) <- cts
print(sapply(mats, dim))

donor_meta <- read.csv(file.path(BASE, "donor_meta.csv"), row.names = 1)
str(donor_meta[, c("Source","Outcome","Death28","Institute","Age","Hospitalstay","TimeSinceOnset")])

      CD4  CD8    B   NK cMono ncMono   DC   PB  GDT   DP
[1,]  121  119  109  118   120    105   67   52   50   57
[2,] 3000 3000 3000 3000  3000   3000 3000 3000 3000 3000


'data.frame':	124 obs. of  7 variables:
 $ Source        : chr  "COVID_SEV" "COVID_MILD" "COVID_HCW_MILD" "COVID_CRIT" ...
 $ Outcome       : chr  "Intubated, ventilated" "Hospitalized, no O2" "Not hospitalized" "Intubated, ventilated" ...
 $ Death28       : chr  "False" "False" "False" "False" ...
 $ Institute     : chr  "Oxford" "Oxford" "Oxford" "Oxford" ...
 $ Age           : chr  "51-60" "51-60" "41-50" "71-80" ...
 $ Hospitalstay  : num  1 2 NA 4 2 NA 3 -12 NA NA ...
 $ TimeSinceOnset: num  12 12 17 14 6 15 23 9 NA 6 ...


In [55]:
donors_per_ct <- lapply(mats, rownames)
common_donors <- Reduce(intersect, donors_per_ct)
cat("donors per CT:", sapply(donors_per_ct, length), "\n")
cat("common to all 10 CTs:", length(common_donors), "\n")
cat("union:", length(unique(unlist(donors_per_ct))), "\n")

if (is.character(donor_meta$Age)) {
  suppressWarnings(donor_meta$Age <- as.numeric(donor_meta$Age))
}
cat("Age NAs:", sum(is.na(donor_meta$Age)), "of", nrow(donor_meta), "\n")
print(table(donor_meta$Source, useNA = "ifany"))

donors per CT: 121 119 109 118 120 105 67 52 50 57 


common to all 10 CTs: 9 


union: 122 


Age NAs: 124 of 124 



    COVID_CRIT COVID_HCW_MILD      COVID_LDN     COVID_MILD      COVID_SEV 
            17             13              2             17             30 
           Flu             HV         Sepsis 
            12             10             23 


In [56]:
## Sanity: canonical marker expression per CT matrix (mean logCPM)
markers <- c(CD3D="CD3D", CD4="CD4", CD8A="CD8A", CD14="CD14", LST1="LST1",
             MS4A1="MS4A1", GNLY="GNLY", NCAM1="NCAM1", SDC1="SDC1", FCER1A="FCER1A")
avail <- lapply(mats, function(m) intersect(markers, colnames(m)))
mk <- sapply(names(mats), function(ct) {
  m <- mats[[ct]]
  sapply(markers, function(g) if (g %in% colnames(m)) round(mean(m[, g]), 2) else NA)
})
rownames(mk) <- names(markers)
print(mk)

       CD4 CD8  B NK cMono ncMono DC PB GDT DP
CD3D    NA  NA NA NA    NA     NA NA NA  NA NA
CD4     NA  NA NA NA    NA     NA NA NA  NA NA
CD8A    NA  NA NA NA    NA     NA NA NA  NA NA
CD14    NA  NA NA NA    NA     NA NA NA  NA NA
LST1    NA  NA NA NA    NA     NA NA NA  NA NA
MS4A1   NA  NA NA NA    NA     NA NA NA  NA NA
GNLY    NA  NA NA NA    NA     NA NA NA  NA NA
NCAM1   NA  NA NA NA    NA     NA NA NA  NA NA
SDC1    NA  NA NA NA    NA     NA NA NA  NA NA
FCER1A  NA  NA NA NA    NA     NA NA NA  NA NA


In [57]:
cat("colnames example:", head(colnames(mats[[1]])), "\n")
var <- read.csv(file.path(BASE, "pb_var.csv"))
cat("var columns:", paste(head(names(var)), collapse=", "), "\n")
print(head(var[, intersect(c("feature_id","feature_name"), names(var))]))

## build ID -> symbol map (used for all downstream interpretation)
id2sym <- setNames(var$feature_name, var$feature_id)
saveRDS(id2sym, "/workspace/benchmarks/combat/id2sym.rds")

## redo marker check on symbols
mk <- sapply(names(mats), function(ct) {
  m <- mats[[ct]]
  syms <- id2sym[colnames(m)]
  sapply(markers, function(g) {
    j <- which(syms == g)[1]
    if (is.na(j)) NA else round(mean(m[, j]), 2)
  })
})
rownames(mk) <- names(markers)
print(mk)

colnames example: ENSG00000134184 ENSG00000114374 ENSG00000067048 ENSG00000183878 ENSG00000012817 ENSG00000129824 


var columns: feature_id, feature_types, feature_is_filtered, feature_name, feature_reference, feature_biotype 


       feature_id    feature_name
1 ENSG00000243485     MIR1302-2HG
2 ENSG00000237613         FAM138A
3 ENSG00000186092           OR4F5
4 ENSG00000239945 ENSG00000239945
5 ENSG00000239906 ENSG00000239906
6 ENSG00000241860 ENSG00000241860


        CD4  CD8    B   NK cMono ncMono   DC   PB  GDT   DP
CD3D     NA   NA   NA   NA    NA     NA 2.98 2.22   NA   NA
CD4      NA 1.76 1.55 1.82  6.93     NA   NA   NA 1.77   NA
CD8A   3.95   NA   NA   NA    NA   2.37 1.53 1.53   NA   NA
CD14   4.11   NA 3.75 3.90    NA     NA   NA 2.21 3.53 3.10
LST1     NA   NA   NA   NA    NA     NA   NA 2.30   NA   NA
MS4A1    NA 4.32   NA 2.83    NA   2.16 2.49 3.30   NA 3.43
GNLY   7.28   NA   NA   NA    NA     NA   NA   NA   NA   NA
NCAM1  1.14 2.79   NA   NA  1.41     NA   NA   NA   NA 1.29
SDC1     NA   NA   NA   NA    NA     NA   NA   NA   NA   NA
FCER1A   NA   NA   NA   NA  1.87     NA   NA   NA   NA   NA


In [58]:
## ---- donor metadata prep (reload fresh: kernel copy had Age overwritten) ----
meta <- read.csv(file.path(BASE, "donor_meta.csv"), row.names = 1)
age_map <- c("21-30"=25.5, "31-40"=35.5, "41-50"=45.5, "51-60"=55.5,
             "61-70"=65.5, "71-80"=75.5, "81-90"=85.5, "90+"=92.5)
meta$Age_num <- unname(age_map[as.character(meta$Age)])
cat("Age bands:", paste(sort(unique(meta$Age)), collapse=", "), "-> NAs after map:", sum(is.na(meta$Age_num)), "\n")
meta$Death28 <- meta$Death28 == "True"
meta$Source <- factor(meta$Source)
meta$Outcome <- factor(meta$Outcome)
meta$Institute <- factor(meta$Institute)
meta$GEX_region <- factor(meta$GEX_region)

CAT_BIO <- c("Source", "Outcome", "Death28", "sex")
CONT_BIO <- c("Age_num", "Hospitalstay", "TimeSinceOnset")
CAT_TECH <- c("Institute", "GEX_region")

## ---- baseline representations ----
comp <- as.matrix(read.csv(file.path(BASE, "composition.csv"), row.names = 1))
donors <- sort(unique(unlist(lapply(mats, rownames))))

align_donors <- function(sc, donors) {
  out <- matrix(0, length(donors), ncol(sc), dimnames = list(donors, colnames(sc)))
  common <- intersect(rownames(sc), donors)
  out[common, ] <- sc[common, , drop = FALSE]
  out
}

reps <- list()
reps$random <- rep_random(donors, 10L)
reps$composition <- align_donors(rep_composition(comp), donors)
reps$clr_composition <- align_donors(rep_clr(comp), donors)
reps$global_pca <- align_donors(.safe_pca(as.matrix(read.csv(file.path(BASE, "mat", "global.csv"), row.names = 1)), 10L), donors)
reps$perct_pca <- do.call(cbind, lapply(cts, function(ct)
  align_donors(.safe_pca(mats[[ct]], 3L), donors)))
groups <- list(T = c("CD4","CD8","GDT","DP"), B_PB = c("B","PB"), NK = "NK",
               Mono = c("cMono","ncMono"), DC = "DC")
reps$grouped_pca <- do.call(cbind, lapply(names(groups), function(gn)
  align_donors(.safe_pca(as.matrix(read.csv(file.path(BASE, "mat", paste0("group_", gn, ".csv")), row.names = 1)), 5L), donors)))

print(sapply(reps, dim))

Age bands: >=91, 19-30, 31-40, 41-50, 51-60, 61-70, 71-80, 81-90 -> NAs after map: 7 


     random composition clr_composition global_pca perct_pca grouped_pca
[1,]    122         122             122        122       122         122
[2,]     10          18              18         10        30          25


In [59]:
## ---- evaluate baselines: Panel A (biology) + Panel B (technical) ----
meta_al <- meta[donors, , drop = FALSE]

eval_baselines <- do.call(rbind, lapply(names(reps), function(rn) {
  ev <- evaluate_representation(reps[[rn]], meta_al,
                                cat_cols = c(CAT_BIO, CAT_TECH),
                                cont_cols = CONT_BIO)
  data.frame(representation = rn, ev)
}))
eval_baselines$panel <- ifelse(eval_baselines$covariate %in% CAT_TECH, "B_technical", "A_biology")
print(eval_baselines[order(eval_baselines$covariate, -eval_baselines$value), ], row.names = FALSE)
saveRDS(eval_baselines, "/workspace/benchmarks/combat/eval_baselines.rds")

  representation      covariate        type           metric     value
       perct_pca        Age_num  continuous max_abs_spearman 0.3943529
      global_pca        Age_num  continuous max_abs_spearman 0.3736313
     composition        Age_num  continuous max_abs_spearman 0.3641427
     grouped_pca        Age_num  continuous max_abs_spearman 0.3634354
 clr_composition        Age_num  continuous max_abs_spearman 0.2865260
          random        Age_num  continuous max_abs_spearman 0.2573811
     grouped_pca        Death28 categorical     knn_macro_f1 0.5766210
     composition        Death28 categorical     knn_macro_f1 0.5719298
 clr_composition        Death28 categorical     knn_macro_f1 0.5571215
      global_pca        Death28 categorical     knn_macro_f1 0.5148538
       perct_pca        Death28 categorical     knn_macro_f1 0.4874886
          random        Death28 categorical     knn_macro_f1 0.4601770
     composition     GEX_region categorical     knn_macro_f1 0.3535025
 clr_c

In [60]:
## fix age map (bands are '19-30' and '>=91') and re-evaluate
age_map <- c("19-30"=24.5, "31-40"=35.5, "41-50"=45.5, "51-60"=55.5,
             "61-70"=65.5, "71-80"=75.5, "81-90"=85.5, ">=91"=92.5)
meta$Age_num <- unname(age_map[as.character(meta$Age)])
cat("Age NAs:", sum(is.na(meta$Age_num)), "\n")
meta_al <- meta[donors, , drop = FALSE]

eval_baselines <- do.call(rbind, lapply(names(reps), function(rn) {
  ev <- evaluate_representation(reps[[rn]], meta_al,
                                cat_cols = c(CAT_BIO, CAT_TECH),
                                cont_cols = CONT_BIO)
  data.frame(representation = rn, ev)
}))
eval_baselines$panel <- ifelse(eval_baselines$covariate %in% CAT_TECH, "B_technical", "A_biology")
saveRDS(eval_baselines, "/workspace/benchmarks/combat/eval_baselines.rds")

## compact wide view
wide <- reshape(eval_baselines[, c("representation","covariate","value")],
                idvar = "representation", timevar = "covariate", direction = "wide")
rownames(wide) <- wide$representation
wide <- wide[, -1]
names(wide) <- sub("^value\\.", "", names(wide))
print(round(wide[order(rownames(wide)), ], 3))

Age NAs: 0 


                Source Outcome Death28   sex Institute GEX_region Age_num
clr_composition  0.235   0.308   0.557 0.519     0.599      0.311   0.308
composition      0.181   0.178   0.572 0.515     0.538      0.354   0.356
global_pca       0.471   0.359   0.515 0.819     0.538      0.244   0.304
grouped_pca      0.520   0.401   0.577 0.550     0.599      0.190   0.300
perct_pca        0.509   0.336   0.487 0.525     0.673      0.252   0.422
random           0.113   0.167   0.460 0.465     0.939      0.195   0.200
                Hospitalstay TimeSinceOnset
clr_composition        0.318          0.250
composition            0.288          0.252
global_pca             0.306          0.377
grouped_pca            0.441          0.511
perct_pca              0.387          0.493
random                 0.229          0.207


In [61]:
## re-evaluate with fixed macro-F1
source("/workspace/benchmarks/R/metrics.R")  # reload fixed version
eval_baselines <- do.call(rbind, lapply(names(reps), function(rn) {
  ev <- evaluate_representation(reps[[rn]], meta_al,
                                cat_cols = c(CAT_BIO, CAT_TECH),
                                cont_cols = CONT_BIO)
  data.frame(representation = rn, ev)
}))
eval_baselines$panel <- ifelse(eval_baselines$covariate %in% CAT_TECH, "B_technical", "A_biology")
saveRDS(eval_baselines, "/workspace/benchmarks/combat/eval_baselines.rds")

wide <- reshape(eval_baselines[, c("representation","covariate","value")],
                idvar = "representation", timevar = "covariate", direction = "wide")
rownames(wide) <- wide$representation; wide <- wide[, -1]
names(wide) <- sub("^value\\.", "", names(wide))
print(round(wide[order(rownames(wide)), ], 3))

                Source Outcome Death28   sex Institute GEX_region Age_num
clr_composition  0.235   0.308   0.557 0.519     0.599      0.249   0.308
composition      0.158   0.178   0.572 0.515     0.538      0.283   0.356
global_pca       0.412   0.359   0.515 0.819     0.538      0.244   0.304
grouped_pca      0.455   0.401   0.577 0.550     0.599      0.152   0.300
perct_pca        0.446   0.336   0.487 0.525     0.673      0.202   0.422
random           0.099   0.167   0.460 0.465     0.470      0.156   0.200
                Hospitalstay TimeSinceOnset
clr_composition        0.318          0.250
composition            0.288          0.252
global_pca             0.306          0.377
grouped_pca            0.441          0.511
perct_pca              0.387          0.493
random                 0.229          0.207


In [62]:
# Candidate alternative IBD collections on CELLxGENE
ibd = allc[allc.collection_name.str.contains("Crohn|colitis|inflammatory bowel|IBD", case=False, na=False)]
print(ibd.to_string(index=False))
print()
for cid in ibd.collection_id:
    sub = ds[ds.collection_id == cid]
    print(sub[["dataset_id","dataset_title","dataset_total_cell_count"]].to_string(index=False), "\n")

                       collection_id                                                                                                                        collection_name
2b02dff7-e427-4cdc-96fb-c0f354c099aa Single-Cell Analysis of Crohn’s Disease Lesions Identifies a Pathogenic Cellular Module Associated with Resistance to Anti-TNF Therapy
5c868b6f-62c5-4532-9d7f-a346ad4b50a7  The landscape of immune dysregulation in Crohn’s disease revealed through single-cell transcriptomic profiling in the ileum and colon
17481d16-ee44-49e5-bcf0-28c0780d8c4a                              Single-Cell Sequencing of Developing Human Gut Reveals Transcriptional Links to Childhood Crohn’s Disease
33d19f34-87f5-455b-8ca5-9023a2e5453d                                                        Intra- and Inter-cellular Rewiring of the Human Colon during Ulcerative Colitis

                          dataset_id dataset_title  dataset_total_cell_count
36c867a7-be10-4e69-9b39-5de12b0af6da         Ileum            

In [63]:
import cellxgene_census, pandas as pd
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    for name, dsids in [("martin_antitnf", ["36c867a7-be10-4e69-9b39-5de12b0af6da"]),
                        ("kong_crohns", ["518d9049-2a76-44f8-8abc-1e2b59ab5ba1",
                                         "5ce42b38-d867-487f-9b40-e8bb00b21d0b"])]:
        ids = "','".join(dsids)
        o = (census["census_data"]["homo_sapiens"].obs.read(
            value_filter=f"dataset_id in ['{ids}'] and is_primary_data == True",
            column_names=["donor_id","disease","cell_type","tissue"]).concat().to_pandas())
        dm = o.drop_duplicates("donor_id")
        print(f"== {name}: {len(o)} cells, {o.donor_id.nunique()} donors ==")
        print(dm.groupby("disease").size().to_string())
        ct = pd.crosstab(o.donor_id, o.cell_type)
        well = (ct >= 50).sum()
        print(f"cell types with >=30 donors at >=50 cells: {sorted(well[well>=30].index.tolist())}")
        print()

== martin_antitnf: 0 cells, 0 donors ==
disease
Alzheimer disease                                                                         0
B-cell acute lymphoblastic leukemia                                                       0
B-cell non-Hodgkin lymphoma                                                               0
Barrett esophagus                                                                         0
COVID-19                                                                                  0
Crohn disease                                                                             0
Down syndrome                                                                             0
HER2 positive breast carcinoma                                                            0
HIV infectious disease                                                                    0
HIV infectious disease || leishmaniasis                                                   0
HIV infectious disease || viscer

/tmp/ipykernel_5096/2995396861.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(dm.groupby("disease").size().to_string())


== kong_crohns: 168979 cells, 22 donors ==
disease
Alzheimer disease                                                                          0
B-cell acute lymphoblastic leukemia                                                        0
B-cell non-Hodgkin lymphoma                                                                0
Barrett esophagus                                                                          0
COVID-19                                                                                   0
Crohn disease                                                                              5
Down syndrome                                                                              0
HER2 positive breast carcinoma                                                             0
HIV infectious disease                                                                     0
HIV infectious disease || leishmaniasis                                                    0
HIV infectious dise

/tmp/ipykernel_5096/2995396861.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(dm.groupby("disease").size().to_string())


cell types with >=30 donors at >=50 cells: []



In [64]:
def obs_counts(dsids, primary=True):
    ids = "','".join(dsids)
    filt = f"dataset_id in ['{ids}']" + (" and is_primary_data == True" if primary else "")
    with cellxgene_census.open_soma(census_version="2025-11-08") as census:
        o = (census["census_data"]["homo_sapiens"].obs.read(
            value_filter=filt,
            column_names=["donor_id","disease","cell_type"]).concat().to_pandas())
    for c in ["disease","cell_type"]:
        o[c] = o[c].astype(str)
    return o

# Martin: try without primary filter
o1 = obs_counts(["36c867a7-be10-4e69-9b39-5de12b0af6da"], primary=False)
print(f"martin (no primary filter): {len(o1)} cells, {o1.donor_id.nunique()} donors")
if len(o1):
    print(o1.drop_duplicates("donor_id")["disease"].value_counts().to_string())

# Kong colon immune + epithelial
o2 = obs_counts(["518d9049-2a76-44f8-8abc-1e2b59ab5ba1","5ce42b38-d867-487f-9b40-e8bb00b21d0b"])
print(f"\nkong colon (immune+epithelial): {len(o2)} cells, {o2.donor_id.nunique()} donors")
if len(o2):
    print(o2.drop_duplicates("donor_id")["disease"].value_counts().to_string())
    ct = pd.crosstab(o2.donor_id, o2.cell_type)
    well = (ct >= 50).sum()
    print(f"CTs with >=25 donors at >=50 cells: {sorted(well[well>=25].index.tolist())}")

martin (no primary filter): 32458 cells, 11 donors
disease
normal    11



kong colon (immune+epithelial): 168979 cells, 22 donors
disease
normal           17
Crohn disease     5
CTs with >=25 donors at >=50 cells: []


In [67]:
## ---- evaluate distributional comparators (finite matrices: GMM + PILOT) ----
dist_reps <- list(gloscope_gmm = gs_gmm, pilot = pilot)
eval_dist <- do.call(rbind, lapply(names(dist_reps), function(rn) {
  D <- dist_reps[[rn]]
  dn <- intersect(rownames(D), rownames(meta))
  ev <- evaluate_representation(D[dn, dn], meta[dn, , drop = FALSE],
                                cat_cols = c(CAT_BIO, CAT_TECH),
                                cont_cols = CONT_BIO)
  data.frame(representation = rn, ev)
}))
eval_dist$panel <- ifelse(eval_dist$covariate %in% CAT_TECH, "B_technical", "A_biology")

wide_d <- reshape(eval_dist[, c("representation","covariate","value")],
                  idvar = "representation", timevar = "covariate", direction = "wide")
rownames(wide_d) <- wide_d$representation; wide_d <- wide_d[, -1]
names(wide_d) <- sub("^value\\.", "", names(wide_d))
print(round(wide_d, 3))
saveRDS(eval_dist, "/workspace/benchmarks/combat/eval_dist.rds")

             Source Outcome Death28   sex Institute GEX_region Age_num
gloscope_gmm  0.293    0.29   0.573 0.424     0.468      0.198   0.164
pilot         0.192    0.16   0.466 0.445     0.461      0.195   0.156
             Hospitalstay TimeSinceOnset
gloscope_gmm        0.247          0.397
pilot               0.151          0.227


In [66]:
for (rn in names(dist_reps)) {
  D <- dist_reps[[rn]]
  na_frac <- mean(!is.finite(as.matrix(D)))
  bad_rows <- rownames(D)[rowSums(!is.finite(as.matrix(D))) > 0]
  cat(rn, ": non-finite frac =", round(na_frac, 4),
      "| affected donors:", length(bad_rows), ifelse(length(bad_rows)>0, paste0(" (", paste(head(bad_rows,5), collapse=","), "...)"), ""), "\n")
}

gloscope_knn : non-finite frac = 0.032 | affected donors: 124  (S00056,N00023,S00005,S00043,G05153...) 
gloscope_gmm : non-finite frac = 0 | affected donors: 0  
pilot : non-finite frac = 0 | affected donors: 0  
